In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -*- coding: utf-8 -*-
"""
╔══════════════════════════════════════════════════════════════╗
║   EXPLAINABLE TABNET — FRAMEWORK PREDIKSI RISIKO AKADEMIK   ║
║   SMP NEGERI 1 CIBADAK — VERSI FINAL (tesis_output_v4)      ║
╠══════════════════════════════════════════════════════════════╣
║  Tiga skema evaluasi komplementer:                           ║
║  • Versi B : StratifiedGroupKFold 5-fold (3 kelas)          ║
║  • Versi E : Temporal Split sem 1-5/sem 6 (2 kelas)         ║
║  • Versi G : LOSO Temporal + Binary Classification           ║
╠══════════════════════════════════════════════════════════════╣
║  XAI Framework (4 metode):                                   ║
║  1. Attention Masks TabNet (global, inheren)                 ║
║  2. SHAP KernelExplainer  (global, model-agnostic)          ║
║  3. LIME                  (local per siswa, 3-run avg)       ║
║  4. Counterfactual        (what-if, rekomendasi guru)        ║
╠══════════════════════════════════════════════════════════════╣
║  Bebas data leakage:                                         ║
║  ✅ Scaler fit hanya pada train                              ║
║  ✅ SMOTE hanya pada train                                   ║
║  ✅ Optuna: val split dari train saja                        ║
║  ✅ Tidak ada eval_set X_test di training                    ║
║  ✅ Tidak ada pemindahan sampel train→test                   ║
║  ✅ Test dievaluasi sekali di akhir tiap skema               ║
╚══════════════════════════════════════════════════════════════╝
"""

# ════════════════════════════════════════════════════════════════
# CELL 1: INSTALL & IMPORT
# ════════════════════════════════════════════════════════════════

import subprocess
subprocess.run(["pip", "install",
    "pytorch-tabnet", "xgboost", "lightgbm", "shap",
    "lime", "scikit-learn", "imbalanced-learn", "optuna",
    "matplotlib", "seaborn", "pandas", "torch",
    "scipy", "-q"])

import os, warnings, json, pickle, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from pytorch_tabnet.tab_model import TabNetClassifier
import xgboost as xgb
import lightgbm as lgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.model_selection import (
    StratifiedGroupKFold, StratifiedKFold, train_test_split)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, cohen_kappa_score, roc_curve, auc)
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr

import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import seaborn as sns
import lime, lime.lime_tabular

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"✅ Library siap | Device: {DEVICE}")
print(f"   Framework: Explainable TabNet v4")
print(f"   Skema: Versi B (StratKFold) + E (Temporal) + G (LOSO Binary)")

# ════════════════════════════════════════════════════════════════
# CELL 2: KONFIGURASI PATH
# ════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

# ── Output ke tesis_output_v4 ──
PATH_DATA = '/content/drive/MyDrive/tesis_raport_v3/output/data_mentah_kohort.csv'
PATH_V4   = '/content/drive/MyDrive/tesis_output_v4'

# Sub-folder per skema
PATH_B    = f'{PATH_V4}/versi_B_stratgroupkfold'
PATH_E    = f'{PATH_V4}/versi_E_temporal'
PATH_G    = f'{PATH_V4}/versi_G_loso_binary'
PATH_XAI  = f'{PATH_V4}/xai_combined'  # XAI gabungan

for p in [PATH_V4, PATH_B, PATH_E, PATH_G, PATH_XAI,
          f'{PATH_B}/plots', f'{PATH_B}/models',
          f'{PATH_E}/plots', f'{PATH_E}/models',
          f'{PATH_G}/plots', f'{PATH_G}/models',
          f'{PATH_XAI}/plots']:
    os.makedirs(p, exist_ok=True)

MAPEL_STANDAR = ["PAI","PKN","B_INDO","B_ING","MTK",
                 "IPA","IPS","B_SUN","PJOK","PKRY","SENI"]
TOTAL_HARI = 120
KKM        = 75
SEED_      = 42

# Label untuk masing-masing skema
LABEL_3 = {0:"Risiko Rendah", 1:"Risiko Sedang", 2:"Risiko Tinggi"}
LABEL_2 = {0:"Tidak Berisiko", 1:"Berisiko"}

N_TRIALS = 30   # Optuna trials per model
N_FOLDS  = 5    # K-Fold untuk Versi B

print(f"✅ Path output: {PATH_V4}")

# ════════════════════════════════════════════════════════════════
# CELL 3: LOAD DATA
# ════════════════════════════════════════════════════════════════

print("\n📂 Load data mentah...")
df_long   = pd.read_csv(PATH_DATA)
df_long   = df_long.sort_values(['NISN','POSISI'])
mapel_ada = [m for m in MAPEL_STANDAR if m in df_long.columns]
MAX_POS   = int(df_long['POSISI'].max())

print(f"   Baris     : {len(df_long)}")
print(f"   Siswa     : {df_long['NISN'].nunique()}")
print(f"   Mapel     : {mapel_ada}")
print(f"   Posisi    : {sorted(df_long['POSISI'].unique())}")
print(f"   Sem akhir : {MAX_POS}")

# ════════════════════════════════════════════════════════════════
# CELL 4: FEATURE ENGINEERING (19 fitur sesuai Tabel 3.3 tesis)
# N_SEMESTER tidak masuk FITUR_COLS (revisi dospem)
# ════════════════════════════════════════════════════════════════

def feature_engineering(df_sub, mapel_ada):
    """
    Hitung 19 fitur longitudinal sesuai Tabel 3.3 tesis.
    N_SEMESTER disimpan sebagai metadata, TIDAK masuk FITUR_COLS.
    Bobot IRA=IRK=IRE sama sebagai baseline netral — justifikasi:
    belum ada studi definitif di konteks SMP Indonesia (BAB III).
    """
    rows = []
    for nisn, grp in df_sub.groupby('NISN'):
        grp   = grp.sort_values('POSISI')
        nama  = grp['NAMA'].iloc[-1]
        kelas = (grp['KELAS_TERAKHIR'].iloc[-1]
                 if 'KELAS_TERAKHIR' in grp.columns else '-')
        n_pos = len(grp)
        nm    = grp[mapel_ada].values.astype(float)
        rs    = np.nanmean(nm, axis=1)

        rata   = float(np.nanmean(nm))
        slope  = (float(np.polyfit(np.arange(len(rs)), rs, 1)[0])
                  if len(rs) >= 2 else 0.0)
        std_v  = float(np.nanstd(nm))
        minv   = float(np.nanmin(nm))
        jb     = int(np.nansum(nm < KKM))

        def pct(row):
            a = ((row.get('SAKIT') or 0) +
                 (row.get('IZIN')  or 0) +
                 (row.get('ALPA')  or 0))
            return max(0, (TOTAL_HARI - a) / TOTAL_HARI * 100)

        pl     = [pct(r) for _, r in grp.iterrows()]
        rh     = float(np.mean(pl))
        ta     = float(grp['ALPA'].sum())
        tk     = float((grp['SAKIT']+grp['IZIN']+grp['ALPA']).sum())
        th     = float(pl[-1]-pl[0]) if len(pl) >= 2 else 0.0

        re_    = float(grp['RATA_EKSKUL'].mean())
        je     = float(grp['JML_EKSKUL'].max())
        se     = grp['RATA_EKSKUL'].std()
        ke     = float(0 if pd.isna(se) else se)
        ne     = int((grp['JML_EKSKUL'] == 0).any())

        IRA = max(0, min(4, (100 - rata) / 25))
        IRK = max(0, min(4, (tk / (n_pos * TOTAL_HARI)) * 4))
        IRE = max(0, min(4, 4 - re_))
        SRK = IRA + IRK + IRE

        rec = {
            "NISN": nisn, "NAMA": nama,
            "KELAS_TERAKHIR": kelas,
            "N_SEMESTER": n_pos,           # metadata saja
            "rata_nilai_inti_semua"    : round(rata, 4),
            "tren_nilai"               : round(slope, 4),
            "standar_deviasi_nilai"    : round(std_v, 4),
            "nilai_terendah_semua"     : round(minv, 4),
            "jumlah_mapel_dibawah_75"  : jb,
            "persentase_kehadiran_rata": round(rh, 4),
            "total_alfa_6semester"     : int(ta),
            "total_ketidakhadiran"     : int(tk),
            "tren_kehadiran"           : round(th, 4),
            "rata_nilai_ekskul"        : round(re_, 4),
            "jumlah_ekskul_diikuti"    : int(je),
            "konsistensi_ekskul"       : round(ke, 4),
            "pernah_tidak_ikut_ekskul" : ne,
            "indeks_risiko_akademik"   : round(IRA, 4),
            "indeks_risiko_kehadiran"  : round(IRK, 4),
            "indeks_risiko_ekskul"     : round(IRE, 4),
            "skor_risiko_komposit"     : round(SRK, 4),
        }
        rpp = {}
        for _, row in grp.iterrows():
            p = int(row['POSISI'])
            rpp[f'rata_pos{p}'] = float(
                np.nanmean([row[m] for m in mapel_ada]))
        for p in range(1, 7):
            rec[f'rata_pos{p}'] = rpp.get(f'rata_pos{p}', 0.0)
        rows.append(rec)

    df_f = pd.DataFrame(rows)
    print(f"   ✅ {len(df_f)} siswa × {len(df_f.columns)} kolom")
    return df_f

print("\n⚙️  Feature engineering — populasi penuh...")
df_full = feature_engineering(df_long, mapel_ada)

# Kolom fitur (N_SEMESTER tidak masuk)
FITUR_COLS = [
    "rata_nilai_inti_semua","tren_nilai",
    "standar_deviasi_nilai","nilai_terendah_semua",
    "jumlah_mapel_dibawah_75",
    "persentase_kehadiran_rata","total_alfa_6semester",
    "total_ketidakhadiran","tren_kehadiran",
    "rata_nilai_ekskul","jumlah_ekskul_diikuti",
    "konsistensi_ekskul","pernah_tidak_ikut_ekskul",
    "indeks_risiko_akademik","indeks_risiko_kehadiran",
    "indeks_risiko_ekskul",
    "rata_pos1","rata_pos2","rata_pos3","rata_pos4","rata_pos5",
]
FITUR_COLS = [c for c in FITUR_COLS if c in df_full.columns]
print(f"\n   Fitur model: {len(FITUR_COLS)} (N_SEMESTER tidak termasuk)")

# SRK dan Q25/Q75 populasi penuh
srk_full = df_full['skor_risiko_komposit']
Q25_FULL = float(srk_full.quantile(0.25))
Q75_FULL = float(srk_full.quantile(0.75))

print(f"   Q25 (populasi): {Q25_FULL:.4f}")
print(f"   Q75 (populasi): {Q75_FULL:.4f}")
print(f"   Unique SRK   : {srk_full.nunique()} dari {len(srk_full)}")

# Label populasi untuk referensi
df_full['label_3'] = srk_full.apply(
    lambda x: 0 if x < Q25_FULL else (1 if x < Q75_FULL else 2))
df_full['label_2'] = srk_full.apply(
    lambda x: 0 if x < Q75_FULL else 1)
df_full.to_csv(f"{PATH_V4}/dataset_fitur_full.csv", index=False)
print(f"\n   Dataset lengkap → {PATH_V4}/dataset_fitur_full.csv")

# ════════════════════════════════════════════════════════════════
# CELL 5: FUNGSI-FUNGSI REUSABLE
# ════════════════════════════════════════════════════════════════

# ── Fungsi evaluasi metrik ──
def hitung_metrik(y_true, y_pred, y_proba, nama="",
                  mode='multiclass'):
    acc  = accuracy_score(y_true, y_pred)
    avg  = 'binary' if mode == 'binary' else 'macro'
    f1   = f1_score(y_true, y_pred,
                    average=avg, zero_division=0)
    prec = precision_score(y_true, y_pred,
                           average=avg, zero_division=0)
    rec  = recall_score(y_true, y_pred,
                        average=avg, zero_division=0)
    try:
        if mode == 'binary':
            auc_v = roc_auc_score(y_true, y_proba[:,1])
        else:
            kelas = np.unique(y_true)
            yb    = label_binarize(y_true, classes=[0,1,2])
            auc_v = roc_auc_score(
                yb[:,kelas], y_proba[:,kelas],
                multi_class='ovr', average='macro')
    except:
        auc_v = 0.0
    kappa = cohen_kappa_score(y_true, y_pred)
    cm    = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(cm.shape[0]):
        TP = cm[i,i]; FP = cm[:,i].sum()-TP
        FN = cm[i,:].sum()-TP; TN = cm.sum()-TP-FP-FN
        specs.append(TN/(TN+FP) if (TN+FP)>0 else 0)
    spec = float(np.mean(specs))
    print(f"   [{nama:<20}] Acc={acc:.4f} F1={f1:.4f} "
          f"AUC={auc_v:.4f} κ={kappa:.4f}")
    return {'Model':nama,'Accuracy':acc,'F1':f1,
            'Precision':prec,'Recall':rec,
            'AUC-ROC':auc_v,'Kappa':kappa,'Specificity':spec}

# ── 6 model builders (dipakai di semua skema) ──
def build_tabnet(best_p, n_class=3):
    return TabNetClassifier(
        n_d=best_p.get('n_d',32),
        n_a=best_p.get('n_a',32),
        n_steps=best_p.get('n_steps',5),
        gamma=best_p.get('gamma',1.5),
        lambda_sparse=best_p.get('lambda_sparse',1e-4),
        mask_type=best_p.get('mask_type','entmax'),
        optimizer_fn=torch.optim.Adam,
        optimizer_params={'lr':2e-2},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={'step_size':10,'gamma':0.9},
        seed=SEED, verbose=0)

class ImprovedDNN(nn.Module):
    def __init__(self, inp, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp,256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64,32), nn.BatchNorm1d(32),
            nn.ReLU(), nn.Linear(32, nc))
    def forward(self, x): return self.net(x)

def train_dnn(Xtr, ytr, Xte, yte, nc=3,
              epochs=200, patience=20):
    m   = ImprovedDNN(Xtr.shape[1], nc).to(DEVICE)
    opt = optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs, eta_min=1e-5)
    cnt = np.bincount(ytr, minlength=nc)
    cnt = np.where(cnt==0, 1, cnt)
    wts = torch.FloatTensor([1/c for c in cnt]).to(DEVICE)
    crt = nn.CrossEntropyLoss(weight=wts)
    Xt  = torch.FloatTensor(Xtr).to(DEVICE)
    yt  = torch.LongTensor(ytr).to(DEVICE)
    Xe  = torch.FloatTensor(Xte).to(DEVICE)
    ldr = DataLoader(TensorDataset(Xt,yt),batch_size=64,shuffle=True)
    bf, bs, ni = 0, None, 0
    for ep in range(epochs):
        m.train()
        for xb,yb in ldr:
            opt.zero_grad()
            crt(m(xb),yb).backward()
            nn.utils.clip_grad_norm_(m.parameters(),1.0)
            opt.step()
        sch.step()
        if (ep+1)%10==0:
            m.eval()
            with torch.no_grad():
                lg = m(Xe)
                pr = lg.argmax(1).cpu().numpy()
                pb = torch.softmax(lg,1).cpu().numpy()
            f1v = f1_score(yte,pr,
                average='binary' if nc==2 else 'macro',
                zero_division=0)
            if f1v>bf:
                bf=f1v; ni=0
                bs={k:v.clone() for k,v in m.state_dict().items()}
            else:
                ni+=10
            if ni>=patience:
                print(f"   DNN early stop ep {ep+1}")
                break
    m.load_state_dict(bs); m.eval()
    with torch.no_grad():
        lg=m(Xe); pr=lg.argmax(1).cpu().numpy()
        pb=torch.softmax(lg,1).cpu().numpy()
    return m, pr, pb

class Autoencoder(nn.Module):
    def __init__(self, inp, lat=16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(inp,64),nn.ReLU(),
            nn.Linear(64,32),nn.ReLU(),
            nn.Linear(32,lat))
        self.dec = nn.Sequential(
            nn.Linear(lat,32),nn.ReLU(),
            nn.Linear(32,64),nn.ReLU(),
            nn.Linear(64,inp),nn.Sigmoid())
    def forward(self,x):
        z=self.enc(x); return z,self.dec(z)

def train_ae_lr(Xtr, ytr, Xte, nc=3):
    ae  = Autoencoder(Xtr.shape[1]).to(DEVICE)
    opt = optim.Adam(ae.parameters(),lr=1e-3)
    Xt  = torch.FloatTensor(Xtr).to(DEVICE)
    Xe  = torch.FloatTensor(Xte).to(DEVICE)
    ldr = DataLoader(TensorDataset(Xt),batch_size=64,shuffle=True)
    for _ in range(150):
        ae.train()
        for (b,) in ldr:
            opt.zero_grad()
            _,xr=ae(b); nn.MSELoss()(xr,b).backward(); opt.step()
    ae.eval()
    with torch.no_grad():
        Ztr=ae.enc(Xt).cpu().numpy()
        Zte=ae.enc(Xe).cpu().numpy()
    lr=LogisticRegression(max_iter=1000,random_state=SEED,
                           class_weight='balanced',C=1.0)
    lr.fit(Ztr,ytr)
    return ae, lr, lr.predict(Zte), lr.predict_proba(Zte)

# ── Optuna objectives ──
def make_optuna_tabnet(X_tr, y_tr, nc=3):
    def obj(trial):
        p = {
            'n_d': trial.suggest_categorical('n_d',[16,32,48,64]),
            'n_a': trial.suggest_categorical('n_a',[16,32,48,64]),
            'n_steps': trial.suggest_int('n_steps',3,8),
            'gamma': trial.suggest_float('gamma',1.0,2.0),
            'lambda_sparse': trial.suggest_float(
                'lambda_sparse',1e-5,1e-3,log=True),
            'mask_type': trial.suggest_categorical(
                'mask_type',['sparsemax','entmax']),
        }
        Xv1,Xv2,yv1,yv2 = train_test_split(
            X_tr,y_tr,test_size=0.2,
            random_state=trial.number,stratify=y_tr)
        try:
            tn = build_tabnet(p, nc)
            tn.fit(Xv1,yv1,eval_set=[(Xv2,yv2)],
                   eval_metric=['accuracy'],
                   max_epochs=100,patience=20,
                   batch_size=64,virtual_batch_size=32,
                   num_workers=0,drop_last=False)
            avg = 'binary' if nc==2 else 'macro'
            return f1_score(yv2,tn.predict(Xv2),
                            average=avg,zero_division=0)
        except: return 0.0
    return obj

def make_optuna_xgb(X_tr, y_tr, nc=3):
    avg = 'binary' if nc==2 else 'macro'
    def obj(trial):
        p = {
            'n_estimators': trial.suggest_int('n_estimators',100,600),
            'max_depth': trial.suggest_int('max_depth',3,9),
            'learning_rate': trial.suggest_float(
                'learning_rate',0.01,0.3,log=True),
            'subsample': trial.suggest_float('subsample',0.6,1.0),
            'colsample_bytree': trial.suggest_float(
                'colsample_bytree',0.6,1.0),
            'min_child_weight': trial.suggest_int(
                'min_child_weight',1,10),
            'use_label_encoder':False,
            'eval_metric':'mlogloss' if nc>2 else 'logloss',
            'random_state':SEED,'n_jobs':-1,
        }
        m = xgb.XGBClassifier(**p)
        skf = StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
        sc = []
        for tr,vl in skf.split(X_tr,y_tr):
            m.fit(X_tr[tr],y_tr[tr])
            sc.append(f1_score(y_tr[vl],m.predict(X_tr[vl]),
                               average=avg,zero_division=0))
        return float(np.mean(sc))
    return obj

def make_optuna_rf(X_tr, y_tr, nc=3):
    avg = 'binary' if nc==2 else 'macro'
    def obj(trial):
        p = {
            'n_estimators': trial.suggest_int('n_estimators',100,600),
            'max_depth': trial.suggest_categorical(
                'max_depth',[None,5,10,15,20]),
            'min_samples_split': trial.suggest_int(
                'min_samples_split',2,10),
            'min_samples_leaf': trial.suggest_int(
                'min_samples_leaf',1,5),
            'class_weight':'balanced',
            'random_state':SEED,'n_jobs':-1,
        }
        m = RandomForestClassifier(**p)
        skf = StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
        sc = []
        for tr,vl in skf.split(X_tr,y_tr):
            m.fit(X_tr[tr],y_tr[tr])
            sc.append(f1_score(y_tr[vl],m.predict(X_tr[vl]),
                               average=avg,zero_division=0))
        return float(np.mean(sc))
    return obj

def make_optuna_lgb(X_tr, y_tr, nc=3):
    avg = 'binary' if nc==2 else 'macro'
    def obj(trial):
        p = {
            'n_estimators': trial.suggest_int('n_estimators',100,600),
            'learning_rate': trial.suggest_float(
                'learning_rate',0.01,0.3,log=True),
            'num_leaves': trial.suggest_int('num_leaves',20,100),
            'max_depth': trial.suggest_int('max_depth',3,10),
            'min_child_samples': trial.suggest_int(
                'min_child_samples',5,50),
            'subsample': trial.suggest_float('subsample',0.6,1.0),
            'colsample_bytree': trial.suggest_float(
                'colsample_bytree',0.6,1.0),
            'class_weight':'balanced',
            'random_state':SEED,'n_jobs':-1,'verbose':-1,
        }
        m = lgb.LGBMClassifier(**p)
        skf = StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
        sc = []
        for tr,vl in skf.split(X_tr,y_tr):
            m.fit(X_tr[tr],y_tr[tr])
            sc.append(f1_score(y_tr[vl],m.predict(X_tr[vl]),
                               average=avg,zero_division=0))
        return float(np.mean(sc))
    return obj

def run_optuna(obj_fn, n_trials=N_TRIALS):
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj_fn, n_trials=n_trials,
                   show_progress_bar=False)
    return study.best_params, study.best_value

# ════════════════════════════════════════════════════════════════
# FUNGSI PLOT HELPER
# Ketentuan:
#   - Tidak ada judul utama (suptitle) di gambar
#   - Jika subplot > 1: label (a), (b), (c), ... di pojok kiri atas
#   - Semua gambar 300 dpi
# ════════════════════════════════════════════════════════════════

SUB_LABELS = ['(a)','(b)','(c)','(d)','(e)','(f)',
              '(g)','(h)','(i)','(j)','(k)','(l)']

def add_sublabel(ax, label, fontsize=11):
    """Tambahkan label (a),(b),... di pojok kiri atas subplot."""
    ax.text(0.01, 0.99, label,
            transform=ax.transAxes,
            fontsize=fontsize, fontweight='bold',
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.15',
                      facecolor='white', alpha=0.7,
                      edgecolor='none'))

def plot_bar_metrik(df_m, path_out, fname, std_df=None):
    """
    Bar chart 7 metrik. Tanpa judul utama.
    Setiap subplot berlabel (a)–(g).
    """
    cols_m  = ['Accuracy','F1','Precision','Recall',
               'AUC-ROC','Kappa','Specificity']
    cols_ok = [c for c in cols_m if c in df_m.columns]
    clrs    = ['#2c7bb6','#d7191c','#1a9641',
               '#fdae61','#762a83','#e66101','#4d9221']

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for i, col in enumerate(cols_ok):
        ax  = axes[i//4][i%4]
        v   = df_m[col].values
        nm  = [n.replace(' + ','\n+') for n in df_m.index]
        yerr = (std_df[col].values if std_df is not None
                and col in std_df.columns else None)
        bars = ax.bar(nm, v, color=clrs, alpha=0.85,
                      yerr=yerr, capsize=4,
                      error_kw={'linewidth':1.2})
        ax.set_ylabel(col, fontsize=10)
        ax.set_ylim(0, min(1.1, max(v)*1.3+0.05))
        ax.grid(True, alpha=0.3, axis='y')
        for bar, val in zip(bars, v):
            ax.text(bar.get_x()+bar.get_width()/2,
                    val+0.01, f'{val:.3f}',
                    ha='center', fontsize=7, fontweight='bold')
        ax.tick_params(axis='x', labelsize=7)
        add_sublabel(ax, SUB_LABELS[i])

    for i in range(len(cols_ok), 8):
        axes[i//4][i%4].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{path_out}/{fname}", dpi=300,
                bbox_inches='tight')
    plt.close()
    print(f"   ✅ Bar chart → {fname}")

def plot_cm_all(preds_dict, y_true, label_names,
                path_out, fname):
    """
    Confusion matrix semua model. Tanpa judul utama.
    Label (a)–(f) per subplot.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()
    for i, (nm, (pred, _)) in enumerate(preds_dict.items()):
        labels = sorted(label_names.keys())
        cm_    = confusion_matrix(y_true, pred, labels=labels)
        ticks  = [label_names[l][:10] for l in labels]
        sns.heatmap(cm_, annot=True, fmt='d', cmap='Blues',
                    xticklabels=ticks, yticklabels=ticks,
                    ax=axes[i], cbar=False, linewidths=0.5)
        f1_ = f1_score(y_true, pred,
                       average='macro' if len(labels)>2 else 'binary',
                       zero_division=0)
        axes[i].set_xlabel('Prediksi', fontsize=9)
        axes[i].set_ylabel('Aktual', fontsize=9)
        axes[i].set_title(f'{nm}  (F1={f1_:.3f})',
                          fontsize=10, fontweight='bold')
        add_sublabel(axes[i], SUB_LABELS[i])
    plt.tight_layout()
    plt.savefig(f"{path_out}/{fname}", dpi=300,
                bbox_inches='tight')
    plt.close()
    print(f"   ✅ Confusion matrix → {fname}")

# ════════════════════════════════════════════════════════════════
# FUNGSI XAI — SAMA RATA (ATTENTION, SHAP, LIME, COUNTERFACTUAL)
# Tidak ada yang lebih dominan dari yang lain.
# ════════════════════════════════════════════════════════════════

# ── XAI 1: Attention Masks ──────────────────────────────────────
def xai_attention(tabnet_model, fitur_cols, path_out,
                  prefix=""):
    """
    Global feature importance dari attention masks TabNet.
    Inheren — tidak butuh post-hoc explainer.
    Gambar: bar chart horizontal Top-15, tanpa judul utama, 300dpi.
    """
    fi  = tabnet_model.feature_importances_
    df  = pd.DataFrame({'Fitur': fitur_cols,
                        'Importance': fi}
                       ).sort_values('Importance',
                                      ascending=False)
    df.to_csv(f"{path_out}/{prefix}attention_importance.csv",
              index=False)

    fig, ax = plt.subplots(figsize=(10, 7))
    t15 = df.head(15)
    bars = ax.barh(t15['Fitur'][::-1],
                   t15['Importance'][::-1],
                   color='#2c7bb6', alpha=0.85)
    for bar, val in zip(bars,
                        t15['Importance'][::-1].values):
        ax.text(val + 0.002, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8)
    ax.set_xlabel('Skor Kepentingan Fitur (Attention Masks)',
                  fontsize=10)
    ax.set_ylabel('Fitur', fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(f"{path_out}/{prefix}attention_masks.png",
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ [XAI-1] Attention masks → "
          f"{prefix}attention_masks.png")
    return df

# ── XAI 2: SHAP ─────────────────────────────────────────────────
def xai_shap(tabnet_model, X_tr, X_te, fitur_cols,
             df_imp, path_out, prefix="", nc=3):
    """
    SHAP KernelExplainer — model-agnostic, memvalidasi attention.
    Gambar:
      (a) SHAP global bar chart
      (b) Attention masks bar chart
      → disimpan sebagai satu figure 2-panel untuk perbandingan
    Gambar kedua: SHAP summary (beeswarm), 300dpi, tanpa judul.
    """
    print(f"   [XAI-2] SHAP KernelExplainer ({prefix})...")

    def predict_fn(X):
        return tabnet_model.predict_proba(X.astype(np.float32))

    bg_idx    = np.random.choice(len(X_tr),
                                  min(100, len(X_tr)),
                                  replace=False)
    explainer = shap.KernelExplainer(predict_fn, X_tr[bg_idx])
    n_s       = min(50, len(X_te))
    te_idx    = np.random.choice(len(X_te), n_s, replace=False)
    sv = explainer.shap_values(X_te[te_idx], nsamples=200)

    # ── Normalize SHAP output ke shape (n_samples, n_fitur) ──
    # KernelExplainer bisa return:
    #   list of arrays → multiclass (satu array per kelas)
    #   3D array        → (n_samples, n_fitur, n_kelas)
    #   2D array        → (n_samples, n_fitur) binary/single

    if isinstance(sv, list):
        # Verifikasi tiap elemen list adalah 2D
        sv_clean = []
        for s in sv:
            s = np.array(s)
            if s.ndim == 1:
                s = s.reshape(1, -1)
            sv_clean.append(s)
        abs_mean = np.mean([np.abs(s) for s in sv_clean], axis=0)
        sv_risk  = sv_clean[-1]
    elif isinstance(sv, np.ndarray):
        sv = np.array(sv)
        if sv.ndim == 3:
            # (n_samples, n_fitur, n_kelas) → ambil mean absolute
            abs_mean = np.mean(np.abs(sv), axis=2)
            sv_risk  = sv[:, :, -1]
        elif sv.ndim == 2:
            abs_mean = np.abs(sv)
            sv_risk  = sv
        else:
            abs_mean = np.abs(sv).reshape(1, -1)
            sv_risk  = sv.reshape(1, -1)
    else:
        sv       = np.array(sv)
        abs_mean = np.abs(sv)
        sv_risk  = sv

    # Global importance: mean per fitur → shape (n_fitur,)
    gi = np.mean(abs_mean, axis=0).flatten()

    # Pastikan panjang gi = panjang fitur_cols
    n_fitur = len(fitur_cols)
    if len(gi) != n_fitur:
        print(f"   ⚠️ SHAP gi shape={gi.shape}, "
              f"n_fitur={n_fitur} — menyesuaikan...")
        if len(gi) > n_fitur:
            gi = gi[:n_fitur]
        else:
            gi = np.pad(gi, (0, n_fitur - len(gi)))

    df_shap = pd.DataFrame(
        {'Fitur': fitur_cols, 'SHAP_Importance': gi}
    ).sort_values('SHAP_Importance', ascending=False)
    df_shap.to_csv(
        f"{path_out}/{prefix}shap_importance.csv", index=False)

    # Spearman rank correlation vs attention
    df_shap['rank_shap'] = range(1, len(df_shap)+1)
    df_imp2 = df_imp.copy()
    df_imp2['rank_att'] = range(1, len(df_imp2)+1)
    merged  = df_shap.set_index('Fitur').join(
        df_imp2.set_index('Fitur')[['rank_att']]).dropna()
    corr, pval = spearmanr(
        merged['rank_shap'], merged['rank_att'])
    status = 'VALID' if corr >= 0.7 else 'PERLU DISKUSI'
    print(f"   Spearman ρ={corr:.4f} (p={pval:.4f}) → {status}")
    merged.to_csv(
        f"{path_out}/{prefix}shap_vs_attention.csv")

    # Gambar 1: SHAP (a) vs Attention (b) — 2 panel
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    t15s = df_shap.head(15)
    axes[0].barh(t15s['Fitur'][::-1],
                 t15s['SHAP_Importance'][::-1],
                 color='#d7191c', alpha=0.85)
    axes[0].set_xlabel('Rata-rata |Nilai SHAP|', fontsize=10)
    axes[0].set_ylabel('Fitur', fontsize=10)
    axes[0].grid(True, alpha=0.3, axis='x')
    add_sublabel(axes[0], '(a)')

    t15a = df_imp.head(15)
    axes[1].barh(t15a['Fitur'][::-1],
                 t15a['Importance'][::-1],
                 color='#2c7bb6', alpha=0.85)
    axes[1].set_xlabel('Skor Kepentingan (Attention Masks)',
                       fontsize=10)
    axes[1].set_ylabel('Fitur', fontsize=10)
    axes[1].grid(True, alpha=0.3, axis='x')
    add_sublabel(axes[1], '(b)')

    plt.tight_layout()
    plt.savefig(f"{path_out}/{prefix}shap_vs_attention.png",
                dpi=300, bbox_inches='tight')
    plt.close()

    # Gambar 2: SHAP beeswarm summary
    try:
        fig, ax = plt.subplots(figsize=(10, 8))
        shap.summary_plot(sv_risk, X_te[te_idx],
                          feature_names=fitur_cols,
                          show=False, max_display=15,
                          plot_type="dot")
        ax = plt.gca()
        ax.set_xlabel('Nilai SHAP', fontsize=10)
        plt.tight_layout()
        plt.savefig(f"{path_out}/{prefix}shap_summary.png",
                    dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ [XAI-2] SHAP summary beeswarm tersimpan")
    except Exception as e:
        print(f"   ⚠️ SHAP summary: {e}")

    print(f"   ✅ [XAI-2] SHAP selesai (ρ={corr:.3f})")
    return df_shap, corr, te_idx, sv

# ── XAI 3: LIME ─────────────────────────────────────────────────
def xai_lime(tabnet_model, X_tr, X_te, y_te,
             fitur_cols, label_names, path_out,
             prefix="", nc=3):
    """
    LIME local explainability per siswa.
    Rata-rata 3 run untuk stabilitas.
    Gambar: satu figure multi-panel (a)(b)(c)...
      panel per siswa sampel, tanpa judul utama, 300dpi.
    """
    def predict_fn(X):
        return tabnet_model.predict_proba(X.astype(np.float32))

    results    = []
    kelas_list = (list(label_names.keys()) if nc == 3
                  else [1, 0])
    sampel     = []

    # Kumpulkan sampel: 1 per kelas
    for tc in kelas_list:
        idxs = np.where(y_te == tc)[0]
        if len(idxs) == 0:
            print(f"   ⚠️ Skip LIME kelas "
                  f"{label_names.get(tc,tc)}: tidak ada")
            continue
        sampel.append((tc, idxs[0]))

    if not sampel:
        print("   ⚠️ Tidak ada sampel untuk LIME")
        return results

    n_panel = len(sampel)
    fig, axes = plt.subplots(1, n_panel,
                              figsize=(9*n_panel, 6))
    if n_panel == 1:
        axes = [axes]

    for pi, (tc, idx) in enumerate(sampel):
        inst  = X_te[idx]
        pred  = int(tabnet_model.predict(
            inst.reshape(1,-1))[0])
        proba = tabnet_model.predict_proba(
            inst.reshape(1,-1))[0]

        # 3-run average
        runs = []
        for sr in [SEED, SEED+1, SEED+2]:
            exp = lime.lime_tabular.LimeTabularExplainer(
                X_tr,
                feature_names=fitur_cols,
                class_names=[label_names[k]
                              for k in sorted(label_names)],
                mode='classification',
                discretize_continuous=True,
                random_state=sr)
            ei = exp.explain_instance(
                inst, predict_fn,
                num_features=10, top_labels=1)
            runs.append(dict(ei.as_list(label=pred)))

        all_k = set()
        for r in runs: all_k.update(r.keys())
        avg_lime = sorted(
            [(k, np.mean([r.get(k, 0) for r in runs]))
             for k in all_k],
            key=lambda x: abs(x[1]),
            reverse=True)[:10]

        results.append({
            'idx': int(idx),
            'true_label': int(tc),
            'pred_label': pred,
            'pred_nama' : label_names.get(pred, str(pred)),
            'proba'     : proba.tolist(),
            'lime_vals' : avg_lime,
        })

        # Plot panel
        ax   = axes[pi]
        fs   = [v[0][:35] for v in avg_lime]
        sc   = [v[1] for v in avg_lime]
        cols = ['#d7191c' if s > 0 else '#1a9641'
                for s in sc]
        ax.barh(fs[::-1], sc[::-1],
                color=cols[::-1], alpha=0.85)
        ax.axvline(0, color='black', lw=0.8, ls='--')
        ax.set_xlabel('Skor LIME\n(rata-rata 3 run)',
                      fontsize=9)
        ax.set_ylabel('Fitur', fontsize=9)
        # Sub-judul panel: prediksi + probabilitas
        ax.set_title(
            f'Prediksi: {label_names.get(pred,pred)}\n'
            f'P = {proba[pred]:.3f}',
            fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
        add_sublabel(ax, SUB_LABELS[pi])

    plt.tight_layout()
    plt.savefig(f"{path_out}/{prefix}lime_panel.png",
                dpi=300, bbox_inches='tight')
    plt.close()

    with open(f"{path_out}/{prefix}lime_results.json",
              'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"   ✅ [XAI-3] LIME ({n_panel} sampel) → "
          f"{prefix}lime_panel.png")
    return results

# ── XAI 4: Counterfactual ────────────────────────────────────────
def xai_counterfactual(tabnet_model, X_te, y_te,
                       scaler, fitur_cols, df_imp,
                       label_names, path_out,
                       prefix="", delta=0.15, nc=3):
    """
    Counterfactual (what-if) greedy search berbasis attention rank.
    Validasi dengan SHAP: cek apakah fitur CF konsisten top-5 SHAP.
    Gambar: tabel visual counterfactual per siswa, 300dpi.
    """
    results       = []
    target_kelas  = [nc-1, nc-2] if nc > 1 else [1]
    MAX_SAMPEL    = 3

    for tc in target_kelas:
        idxs = np.where(y_te == tc)[0]
        if len(idxs) == 0: continue
        for idx in idxs[:MAX_SAMPEL]:
            inst = X_te[idx].copy()
            p0   = int(tabnet_model.predict(
                inst.reshape(1,-1))[0])
            if p0 == 0: continue

            skenario = []; x_mod = inst.copy()
            for fn in df_imp['Fitur'].tolist()[:15]:
                if fn not in fitur_cols: continue
                fi    = fitur_cols.index(fn)
                buruk = any(kw in fn.lower() for kw in
                            ['alpa','ketidakhadiran',
                             'risiko','dibawah',
                             'tidak_ikut','terendah'])
                for d in [delta, delta*1.5, delta*2.0]:
                    nv = (max(0.0, x_mod[fi]-d) if buruk
                          else min(1.0, x_mod[fi]+d))
                    x2 = x_mod.copy(); x2[fi] = nv
                    pn = int(tabnet_model.predict(
                        x2.reshape(1,-1))[0])
                    or_ = scaler.inverse_transform(
                        inst.reshape(1,-1))[0]
                    nw_ = scaler.inverse_transform(
                        x2.reshape(1,-1))[0]
                    skenario.append({
                        'fitur'      : fn,
                        'nilai_asal' : round(float(or_[fi]),3),
                        'nilai_baru' : round(float(nw_[fi]),3),
                        'arah'       : 'turunkan' if buruk
                                        else 'naikkan',
                        'rekomendasi': (
                            f"Kurangi {fn.replace('_',' ')}"
                            if buruk else
                            f"Tingkatkan {fn.replace('_',' ')}"),
                        'pred_baru'  : pn,
                        'pred_nama'  : label_names.get(pn,str(pn)),
                    })
                    if pn < p0:
                        x_mod = x2.copy(); break
                if skenario and skenario[-1]['pred_baru']==0:
                    break

            p_akhir = int(tabnet_model.predict(
                x_mod.reshape(1,-1))[0])
            results.append({
                'indeks'       : int(idx),
                'pred_asal'    : p0,
                'pred_asal_nm' : label_names.get(p0,str(p0)),
                'pred_akhir'   : p_akhir,
                'pred_akhir_nm': label_names.get(
                    p_akhir, str(p_akhir)),
                'berhasil'     : p_akhir < p0,
                'skenario'     : skenario[:6],
            })
            st = '✅' if p_akhir < p0 else '⚠️'
            print(f"   CF idx={idx}: "
                  f"{label_names.get(p0,p0)} → "
                  f"{label_names.get(p_akhir,p_akhir)} {st}")

    with open(f"{path_out}/{prefix}counterfactual.json",
              'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    # Validasi CF dengan SHAP
    shap_path = f"{path_out}/{prefix}shap_importance.csv"
    n_valid = 0
    if os.path.exists(shap_path):
        df_s      = pd.read_csv(shap_path)
        top5_shap = df_s.head(5)['Fitur'].tolist()
        for r in results:
            cf_f = [s['fitur'] for s in r['skenario'][:5]]
            if len(set(top5_shap) & set(cf_f)) >= 2:
                n_valid += 1
        pct = n_valid/len(results)*100 if results else 0
        print(f"   Validasi CF-SHAP: {n_valid}/{len(results)} "
              f"konsisten ({pct:.0f}%)")

    # Gambar: tabel visual CF per siswa
    if results:
        n_r  = len(results)
        fig, axes = plt.subplots(
            n_r, 1, figsize=(12, 4.5*n_r))
        if n_r == 1: axes = [axes]

        for pi, r in enumerate(results):
            ax  = axes[pi]
            ax.axis('off')
            scn = r['skenario'][:5]
            if not scn:
                ax.text(0.5, 0.5, 'Tidak ada skenario',
                        ha='center', va='center',
                        transform=ax.transAxes)
                continue

            col_lbls = ['Fitur','Nilai Asal',
                        'Nilai Target','Arah',
                        'Rekomendasi Guru',
                        'Prediksi Baru']
            tbl_data = [
                [s['fitur'], s['nilai_asal'],
                 s['nilai_baru'], s['arah'],
                 s['rekomendasi'][:30],
                 s['pred_nama']]
                for s in scn]

            tbl = ax.table(
                cellText=tbl_data,
                colLabels=col_lbls,
                loc='center',
                cellLoc='center')
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(8)
            tbl.scale(1, 1.6)

            # Warna header
            for j in range(len(col_lbls)):
                tbl[0,j].set_facecolor('#2c7bb6')
                tbl[0,j].set_text_props(
                    color='white', fontweight='bold')

            # Warna baris
            for row_i in range(1, len(tbl_data)+1):
                clr = '#f7fbff' if row_i%2==0 else 'white'
                for j in range(len(col_lbls)):
                    tbl[row_i,j].set_facecolor(clr)
                # Kolom prediksi baru
                pn_val = tbl_data[row_i-1][-1]
                face   = ('#c7e9c0' if pn_val in
                           ['Tidak Berisiko','Risiko Rendah']
                           else '#fcbba1')
                tbl[row_i, len(col_lbls)-1].set_facecolor(face)

            p_asal = r['pred_asal_nm']
            p_akhr = r['pred_akhir_nm']
            berh   = '✅ Berhasil' if r['berhasil'] else '⚠️ Belum'
            ax.set_title(
                f'Siswa idx={r["indeks"]} | '
                f'{p_asal} → {p_akhr}  {berh}',
                fontsize=10, fontweight='bold',
                pad=6)
            add_sublabel(ax, SUB_LABELS[pi])

        plt.tight_layout()
        plt.savefig(f"{path_out}/{prefix}counterfactual_table.png",
                    dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   ✅ [XAI-4] Counterfactual tabel → "
              f"{prefix}counterfactual_table.png")

    return results

# ════════════════════════════════════════════════════════════════
# FUNGSI XAI GABUNGAN — Panggil 4 metode sekaligus
# ════════════════════════════════════════════════════════════════

def run_all_xai(tabnet_model, X_tr, X_te, y_te,
                scaler, fitur_cols, label_names,
                path_out, prefix="", nc=3):
    """
    Jalankan 4 metode XAI secara seimbang:
    1. Attention Masks (global, inheren)
    2. SHAP KernelExplainer (global, model-agnostic)
    3. LIME (local per siswa, 3-run avg)
    4. Counterfactual (what-if, rekomendasi guru)
    """
    print(f"\n{'─'*50}")
    print(f"🔍 XAI FRAMEWORK ({prefix})")
    print(f"   4 metode: Attention | SHAP | LIME | CF")
    print(f"{'─'*50}")

    # 1. Attention Masks
    print(f"\n  [1/4] Attention Masks...")
    df_imp = xai_attention(
        tabnet_model, fitur_cols, path_out, prefix)

    # 2. SHAP
    print(f"\n  [2/4] SHAP KernelExplainer...")
    df_shap, corr, _, _ = xai_shap(
        tabnet_model, X_tr, X_te, fitur_cols,
        df_imp, path_out, prefix, nc)

    # 3. LIME
    print(f"\n  [3/4] LIME...")
    lime_res = xai_lime(
        tabnet_model, X_tr, X_te, y_te,
        fitur_cols, label_names, path_out, prefix, nc)

    # 4. Counterfactual
    print(f"\n  [4/4] Counterfactual...")
    cf_res = xai_counterfactual(
        tabnet_model, X_te, y_te,
        scaler, fitur_cols, df_imp, label_names,
        path_out, prefix, nc=nc)

    # Gambar gabungan: top-5 semua metode dalam 1 figure
    _plot_xai_summary(df_imp, df_shap, lime_res, cf_res,
                      label_names, path_out, prefix, nc)

    print(f"\n  ✅ Semua XAI selesai ({prefix})")
    return df_imp, df_shap, corr, lime_res, cf_res

def _plot_xai_summary(df_imp, df_shap, lime_res, cf_res,
                      label_names, path_out, prefix, nc):
    """
    Gambar ringkasan XAI: 4 panel dalam 1 figure.
    (a) Attention top-10
    (b) SHAP top-10
    (c) LIME satu contoh siswa berisiko
    (d) Bar status counterfactual (berhasil/tidak)
    Tanpa judul utama, 300dpi.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()

    # (a) Attention
    t10 = df_imp.head(10)
    axes[0].barh(t10['Fitur'][::-1],
                 t10['Importance'][::-1],
                 color='#2c7bb6', alpha=0.85)
    axes[0].set_xlabel('Skor Kepentingan', fontsize=9)
    axes[0].set_ylabel('Fitur', fontsize=9)
    axes[0].grid(True, alpha=0.3, axis='x')
    add_sublabel(axes[0], '(a)')

    # (b) SHAP
    t10s = df_shap.head(10)
    axes[1].barh(t10s['Fitur'][::-1],
                 t10s['SHAP_Importance'][::-1],
                 color='#d7191c', alpha=0.85)
    axes[1].set_xlabel('Rata-rata |Nilai SHAP|', fontsize=9)
    axes[1].set_ylabel('Fitur', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='x')
    add_sublabel(axes[1], '(b)')

    # (c) LIME — ambil sampel kelas berisiko
    lime_sample = next(
        (r for r in lime_res if r['pred_label'] == nc-1),
        lime_res[0] if lime_res else None)
    if lime_sample:
        lv = lime_sample['lime_vals'][:8]
        fs = [v[0][:30] for v in lv]
        sc = [v[1] for v in lv]
        cb = ['#d7191c' if s>0 else '#1a9641' for s in sc]
        axes[2].barh(fs[::-1], sc[::-1],
                     color=cb[::-1], alpha=0.85)
        axes[2].axvline(0, color='black', lw=0.8, ls='--')
        axes[2].set_xlabel('Skor LIME', fontsize=9)
        axes[2].set_ylabel('Fitur', fontsize=9)
        axes[2].grid(True, alpha=0.3, axis='x')
    add_sublabel(axes[2], '(c)')

    # (d) Status counterfactual
    if cf_res:
        labels_cf = [f"Siswa {r['indeks']}" for r in cf_res]
        colors_cf = ['#1a9641' if r['berhasil']
                     else '#d7191c' for r in cf_res]
        vals_cf   = [1 if r['berhasil'] else 0 for r in cf_res]
        bars_cf   = axes[3].bar(labels_cf, vals_cf,
                                 color=colors_cf, alpha=0.85)
        axes[3].set_ylim(0, 1.3)
        axes[3].set_yticks([0, 1])
        axes[3].set_yticklabels(['Tidak Berhasil', 'Berhasil'])
        axes[3].set_ylabel('Status', fontsize=9)
        axes[3].grid(True, alpha=0.3, axis='y')
        for bar, r in zip(bars_cf, cf_res):
            lbl = (f"{r['pred_asal_nm'][:8]} →\n"
                   f"{r['pred_akhir_nm'][:8]}")
            axes[3].text(
                bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.03, lbl,
                ha='center', va='bottom',
                fontsize=7, fontweight='bold')
    add_sublabel(axes[3], '(d)')

    plt.tight_layout()
    plt.savefig(f"{path_out}/{prefix}xai_summary_4panel.png",
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Ringkasan XAI 4-panel → "
          f"{prefix}xai_summary_4panel.png")

print("\n✅ Semua fungsi siap (XAI sama rata, 300dpi, label (a)(b)...)")

# ════════════════════════════════════════════════════════════════
# ╔═══════════════════════════════════════════════════╗
# ║  VERSI B: STRATIFIEDGROUPKFOLD 5-FOLD (3 KELAS)  ║
# ╚═══════════════════════════════════════════════════╝
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("VERSI B — StratifiedGroupKFold 5-fold | 3 Kelas")
print("="*60)

# ── Data ──
X_B    = df_full[FITUR_COLS].fillna(0).values.astype(np.float32)
y_B    = df_full['label_3'].values.astype(int)
grp_B  = df_full['NISN'].values

print(f"\n   Siswa: {len(X_B)} | Distribusi: {np.bincount(y_B)}")

# ── Optuna tuning (inner CV dari seluruh data) ──
print(f"\n🔍 Optuna Tuning Versi B ({N_TRIALS} trials/model)...")

# Pakai scaler sementara untuk tuning
sc_tmp = MinMaxScaler()
X_B_sc = sc_tmp.fit_transform(X_B).astype(np.float32)

sm_tmp = SMOTE(random_state=SEED, k_neighbors=5)
X_B_sm, y_B_sm = sm_tmp.fit_resample(X_B_sc, y_B)

print("  🔵 TabNet...")
bp_B_tn, _ = run_optuna(make_optuna_tabnet(X_B_sm, y_B_sm, 3))
print(f"     Best: {bp_B_tn}")

print("  🟠 XGBoost...")
bp_B_xgb, _ = run_optuna(make_optuna_xgb(X_B_sm, y_B_sm, 3))
print(f"     Best: {bp_B_xgb}")

print("  🟢 Random Forest...")
bp_B_rf, _  = run_optuna(make_optuna_rf(X_B_sm, y_B_sm, 3))
print(f"     Best: {bp_B_rf}")

print("  🟡 LightGBM...")
bp_B_lgb, _ = run_optuna(make_optuna_lgb(X_B_sm, y_B_sm, 3))
print(f"     Best: {bp_B_lgb}")

bp_B_all = {'TabNet':bp_B_tn,'XGBoost':bp_B_xgb,
             'RandomForest':bp_B_rf,'LightGBM':bp_B_lgb}
with open(f"{PATH_B}/best_params.json",'w') as f:
    json.dump(bp_B_all, f, indent=2)

# ── 5-fold CV ──
print(f"\n🔁 5-fold StratifiedGroupKFold...")
sgkf  = StratifiedGroupKFold(
    n_splits=N_FOLDS, shuffle=True, random_state=SEED)
MODEL_NAMES_B = ['TabNet','XGBoost','Random Forest',
                  'LightGBM','Improved DNN','Autoencoder+LR']
fold_res_B = {nm:[] for nm in MODEL_NAMES_B}
last_fold  = {}

for fold,(tr_i,te_i) in enumerate(sgkf.split(X_B,y_B,grp_B)):
    print(f"\n   ── Fold {fold+1}/{N_FOLDS} ──")
    sc_f = MinMaxScaler()
    Xtr  = sc_f.fit_transform(X_B[tr_i]).astype(np.float32)
    Xte  = sc_f.transform(X_B[te_i]).astype(np.float32)
    ytr  = y_B[tr_i]; yte = y_B[te_i]

    mn_s = min((ytr==c).sum() for c in np.unique(ytr))
    k_s  = max(1, min(5, mn_s-1))
    sm   = SMOTE(random_state=SEED, k_neighbors=k_s)
    Xtr_s, ytr_s = sm.fit_resample(Xtr, ytr)

    # TabNet
    Xv1,Xv2,yv1,yv2 = train_test_split(
        Xtr_s,ytr_s,test_size=0.15,
        random_state=SEED,stratify=ytr_s)
    tn = build_tabnet(bp_B_tn, 3); tn.verbose=0
    tn.fit(Xv1,yv1,eval_set=[(Xv2,yv2)],
           eval_metric=['accuracy'],max_epochs=150,patience=25,
           batch_size=64,virtual_batch_size=32,
           num_workers=0,drop_last=False)
    pp = tn.predict(Xte); pb = tn.predict_proba(Xte)
    m  = hitung_metrik(yte,pp,pb,'TabNet')
    fold_res_B['TabNet'].append(m)

    # XGBoost
    xp = {k:v for k,v in bp_B_xgb.items()}
    xp.update({'use_label_encoder':False,'eval_metric':'mlogloss',
                'random_state':SEED,'n_jobs':-1})
    xm = xgb.XGBClassifier(**xp); xm.fit(Xtr_s,ytr_s)
    pp = xm.predict(Xte); pb = xm.predict_proba(Xte)
    fold_res_B['XGBoost'].append(hitung_metrik(yte,pp,pb,'XGBoost'))

    # RF
    rp = {k:v for k,v in bp_B_rf.items()}
    rp.update({'class_weight':'balanced','random_state':SEED,'n_jobs':-1})
    rm = RandomForestClassifier(**rp); rm.fit(Xtr_s,ytr_s)
    pp = rm.predict(Xte); pb = rm.predict_proba(Xte)
    fold_res_B['Random Forest'].append(hitung_metrik(yte,pp,pb,'Random Forest'))

    # LightGBM
    lp = {k:v for k,v in bp_B_lgb.items()}
    lp.update({'class_weight':'balanced','random_state':SEED,'n_jobs':-1,'verbose':-1})
    lm = lgb.LGBMClassifier(**lp); lm.fit(Xtr_s,ytr_s)
    pp = lm.predict(Xte); pb = lm.predict_proba(Xte)
    fold_res_B['LightGBM'].append(hitung_metrik(yte,pp,pb,'LightGBM'))

    # DNN
    dm,pp,pb = train_dnn(Xtr_s,ytr_s,Xte,yte,nc=3)
    fold_res_B['Improved DNN'].append(hitung_metrik(yte,pp,pb,'Improved DNN'))

    # AE+LR
    _,_,pp,pb = train_ae_lr(Xtr_s,ytr_s,Xte,nc=3)
    fold_res_B['Autoencoder+LR'].append(hitung_metrik(yte,pp,pb,'Autoencoder+LR'))

    # Simpan fold terakhir untuk XAI
    if fold == N_FOLDS-1:
        last_fold = {'Xtr':Xtr_s,'ytr':ytr_s,
                     'Xte':Xte,'yte':yte,
                     'sc':sc_f,'tn':tn,'te_idx':te_i}

# ── Agregasi ──
mk_cols = ['Accuracy','F1','Precision','Recall',
           'AUC-ROC','Kappa','Specificity']
rows_mean, rows_std = [], []
for nm in MODEL_NAMES_B:
    fm = fold_res_B[nm]
    rows_mean.append({'Model':nm,
        **{k:round(np.mean([f[k] for f in fm]),4) for k in mk_cols}})
    rows_std.append({'Model':nm,
        **{k:round(np.std([f[k] for f in fm]),4) for k in mk_cols}})

df_B_mean = pd.DataFrame(rows_mean).set_index('Model')
df_B_std  = pd.DataFrame(rows_std).set_index('Model')
df_B_mean.to_csv(f"{PATH_B}/hasil_mean.csv")
df_B_std.to_csv(f"{PATH_B}/hasil_std.csv")

print(f"\n📊 VERSI B — HASIL ({N_FOLDS}-fold mean±std)")
best_B = df_B_mean['F1'].idxmax()
for nm in MODEL_NAMES_B:
    mk = '★' if nm==best_B else ' '
    print(f"  {mk} {nm:<20} "
          f"F1={df_B_mean.loc[nm,'F1']:.4f}"
          f"±{df_B_std.loc[nm,'F1']:.3f}  "
          f"AUC={df_B_mean.loc[nm,'AUC-ROC']:.4f}"
          f"±{df_B_std.loc[nm,'AUC-ROC']:.3f}")

plot_bar_metrik(df_B_mean,
    f"{PATH_B}/plots","perbandingan_6model.png",
    std_df=df_B_std)

# ── XAI Versi B (dari fold terakhir) ──
print("\n🔍 XAI Versi B...")
tn_B  = last_fold['tn']
Xtr_B = last_fold['Xtr']; ytr_B = last_fold['ytr']
Xte_B = last_fold['Xte']; yte_B = last_fold['yte']
sc_B  = last_fold['sc']

df_imp_B, df_shap_B, corr_B, _, _ = run_all_xai(
    tn_B, Xtr_B, Xte_B, yte_B,
    sc_B, FITUR_COLS, LABEL_3,
    f"{PATH_B}/plots", prefix="B_", nc=3)

# Simpan model final (retrain pada semua data)
print("\n   Training TabNet final (semua data) untuk dashboard...")
sc_B_final = MinMaxScaler()
X_B_final  = sc_B_final.fit_transform(X_B).astype(np.float32)
sm_f       = SMOTE(random_state=SEED, k_neighbors=5)
Xbf, ybf   = sm_f.fit_resample(X_B_final, y_B)
tn_B_final = build_tabnet(bp_B_tn, 3); tn_B_final.verbose=0
Xv1,Xv2,yv1,yv2 = train_test_split(
    Xbf,ybf,test_size=0.15,random_state=SEED,stratify=ybf)
tn_B_final.fit(Xv1,yv1,eval_set=[(Xv2,yv2)],
    eval_metric=['accuracy'],max_epochs=150,patience=25,
    batch_size=64,virtual_batch_size=32,
    num_workers=0,drop_last=False)
tn_B_final.save_model(f"{PATH_B}/models/tabnet_final")
with open(f"{PATH_B}/models/scaler_final.pkl",'wb') as f:
    pickle.dump(sc_B_final, f)
print("   ✅ Model final Versi B tersimpan")

# ════════════════════════════════════════════════════════════════
# ╔═══════════════════════════════════════════════════╗
# ║  VERSI E: TEMPORAL SPLIT SEM 1-5/SEM 6 (3→2 KLS) ║
# ╚═══════════════════════════════════════════════════╝
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("VERSI E — Temporal Split | sem 1-5 train | sem 6 test")
print("="*60)

# ── Split ──
df_tr_E = df_long[df_long['POSISI'] < MAX_POS].copy()
nisn_te_E = set(df_long[df_long['POSISI']==MAX_POS]['NISN'].unique())
df_te_E = df_long[df_long['NISN'].isin(nisn_te_E)].copy()

print(f"\n   Feature engineering train (sem 1-{MAX_POS-1})...")
df_ft_E_tr = feature_engineering(df_tr_E, mapel_ada)
print(f"   Feature engineering test (sem 1-{MAX_POS})...")
df_ft_E_te = feature_engineering(df_te_E, mapel_ada)

# Label: Q25/Q75 dari train saja
srk_E_tr = df_ft_E_tr['skor_risiko_komposit']
Q25_E = float(srk_E_tr.quantile(0.25))
Q75_E = float(srk_E_tr.quantile(0.75))
df_ft_E_tr['proxy_label'] = srk_E_tr.apply(
    lambda x: 0 if x<Q25_E else (1 if x<Q75_E else 2))
df_ft_E_te['proxy_label'] = df_ft_E_te[
    'skor_risiko_komposit'].apply(
    lambda x: 0 if x<Q25_E else (1 if x<Q75_E else 2))

fc_E = [c for c in FITUR_COLS
        if c in df_ft_E_tr.columns
        and c in df_ft_E_te.columns]

Xtr_E_r = df_ft_E_tr[fc_E].fillna(0).values.astype(np.float32)
Xte_E_r = df_ft_E_te[fc_E].fillna(0).values.astype(np.float32)
ytr_E_r = df_ft_E_tr['proxy_label'].values.astype(int)
yte_E   = df_ft_E_te['proxy_label'].values.astype(int)

sc_E = MinMaxScaler()
Xtr_E_s = sc_E.fit_transform(Xtr_E_r).astype(np.float32)
Xte_E   = sc_E.transform(Xte_E_r).astype(np.float32)
with open(f"{PATH_E}/models/scaler.pkl",'wb') as f:
    pickle.dump(sc_E, f)

mn_E = min((ytr_E_r==c).sum() for c in np.unique(ytr_E_r))
sm_E = SMOTE(random_state=SEED, k_neighbors=max(1,min(5,mn_E-1)))
Xtr_E, ytr_E = sm_E.fit_resample(Xtr_E_s, ytr_E_r)

kls_hilang_E = [c for c in range(3) if (yte_E==c).sum()==0]
print(f"\n   Train: {len(Xtr_E)} | Test: {len(Xte_E)}")
print(f"   Distribusi test: {np.bincount(yte_E,minlength=3)}")
if kls_hilang_E:
    print(f"   ⚠️ Kelas hilang di test: "
          f"{[LABEL_3[c] for c in kls_hilang_E]}")
    print(f"      Limitasi temporal split — dicatat di BAB V")

# ── Optuna ──
print(f"\n🔍 Optuna Tuning Versi E...")
bp_E_tn,_  = run_optuna(make_optuna_tabnet(Xtr_E,ytr_E,3))
bp_E_xgb,_ = run_optuna(make_optuna_xgb(Xtr_E,ytr_E,3))
bp_E_rf,_  = run_optuna(make_optuna_rf(Xtr_E,ytr_E,3))
bp_E_lgb,_ = run_optuna(make_optuna_lgb(Xtr_E,ytr_E,3))
bp_E_all = {'TabNet':bp_E_tn,'XGBoost':bp_E_xgb,
             'RandomForest':bp_E_rf,'LightGBM':bp_E_lgb}
with open(f"{PATH_E}/best_params.json",'w') as f:
    json.dump(bp_E_all, f, indent=2)

# ── Training & evaluasi ──
print(f"\n🚀 Training Versi E...")
lbl_E = {k:v for k,v in LABEL_3.items()
         if k not in kls_hilang_E}

# TabNet
Xv1,Xv2,yv1,yv2 = train_test_split(
    Xtr_E,ytr_E,test_size=0.15,
    random_state=SEED,stratify=ytr_E)
tn_E = build_tabnet(bp_E_tn,3); tn_E.verbose=10
tn_E.fit(Xv1,yv1,eval_set=[(Xv2,yv2)],
         eval_metric=['accuracy'],max_epochs=200,patience=30,
         batch_size=64,virtual_batch_size=32,
         num_workers=0,drop_last=False)
tn_E.save_model(f"{PATH_E}/models/tabnet_model")
pp_E_tn  = tn_E.predict(Xte_E)
pb_E_tn  = tn_E.predict_proba(Xte_E)
m_E_tn   = hitung_metrik(yte_E,pp_E_tn,pb_E_tn,"TabNet")

def safe_report(y_true, y_pred, label_names):
    kls = sorted(np.unique(np.concatenate([y_true,y_pred])))
    print(classification_report(y_true,y_pred,
        labels=kls,
        target_names=[label_names.get(k,str(k)) for k in kls],
        zero_division=0))

safe_report(yte_E, pp_E_tn, LABEL_3)

# XGBoost
xp = {k:v for k,v in bp_E_xgb.items()}
xp.update({'use_label_encoder':False,'eval_metric':'mlogloss',
            'random_state':SEED,'n_jobs':-1})
xm_E = xgb.XGBClassifier(**xp); xm_E.fit(Xtr_E,ytr_E)
pp_E_xgb = xm_E.predict(Xte_E); pb_E_xgb = xm_E.predict_proba(Xte_E)
m_E_xgb  = hitung_metrik(yte_E,pp_E_xgb,pb_E_xgb,"XGBoost")
safe_report(yte_E,pp_E_xgb,LABEL_3)
with open(f"{PATH_E}/models/xgboost.pkl",'wb') as f:
    pickle.dump(xm_E,f)

# RF
rp = {k:v for k,v in bp_E_rf.items()}
rp.update({'class_weight':'balanced','random_state':SEED,'n_jobs':-1})
rm_E = RandomForestClassifier(**rp); rm_E.fit(Xtr_E,ytr_E)
pp_E_rf  = rm_E.predict(Xte_E); pb_E_rf = rm_E.predict_proba(Xte_E)
m_E_rf   = hitung_metrik(yte_E,pp_E_rf,pb_E_rf,"Random Forest")
safe_report(yte_E,pp_E_rf,LABEL_3)
with open(f"{PATH_E}/models/random_forest.pkl",'wb') as f:
    pickle.dump(rm_E,f)

# LightGBM
lp = {k:v for k,v in bp_E_lgb.items()}
lp.update({'class_weight':'balanced','random_state':SEED,'n_jobs':-1,'verbose':-1})
lm_E = lgb.LGBMClassifier(**lp); lm_E.fit(Xtr_E,ytr_E)
pp_E_lgb  = lm_E.predict(Xte_E); pb_E_lgb = lm_E.predict_proba(Xte_E)
m_E_lgb   = hitung_metrik(yte_E,pp_E_lgb,pb_E_lgb,"LightGBM")
safe_report(yte_E,pp_E_lgb,LABEL_3)
with open(f"{PATH_E}/models/lightgbm.pkl",'wb') as f:
    pickle.dump(lm_E,f)

# DNN
dm_E,pp_E_dnn,pb_E_dnn = train_dnn(Xtr_E,ytr_E,Xte_E,yte_E,nc=3)
m_E_dnn = hitung_metrik(yte_E,pp_E_dnn,pb_E_dnn,"Improved DNN")
safe_report(yte_E,pp_E_dnn,LABEL_3)
torch.save(dm_E.state_dict(),f"{PATH_E}/models/dnn.pt")

# AE+LR
ae_E,lr_E,pp_E_ae,pb_E_ae = train_ae_lr(Xtr_E,ytr_E,Xte_E,nc=3)
m_E_ae = hitung_metrik(yte_E,pp_E_ae,pb_E_ae,"Autoencoder+LR")
safe_report(yte_E,pp_E_ae,LABEL_3)
torch.save(ae_E.state_dict(),f"{PATH_E}/models/ae.pt")
with open(f"{PATH_E}/models/lr.pkl",'wb') as f:
    pickle.dump(lr_E,f)

# Ringkasan
semua_E = [m_E_tn,m_E_xgb,m_E_rf,m_E_lgb,m_E_dnn,m_E_ae]
df_E = pd.DataFrame(semua_E).set_index('Model').round(4)
df_E.to_csv(f"{PATH_E}/hasil_evaluasi.csv")
best_E = df_E['F1'].idxmax()

print(f"\n📊 VERSI E — HASIL (Temporal Split)")
for nm,row in df_E.iterrows():
    mk = '★' if nm==best_E else ' '
    print(f"  {mk} {nm:<20} F1={row['F1']:.4f} AUC={row['AUC-ROC']:.4f}")
if kls_hilang_E:
    print(f"  ⚠️ Kelas {[LABEL_3[c] for c in kls_hilang_E]} tidak ada di test")

plot_bar_metrik(df_E, f"{PATH_E}/plots",
                "perbandingan_6model.png")

# Buat dictionary prediksi untuk confusion matrix (Versi E)
preds_E = {
    'TabNet': (pp_E_tn, pb_E_tn),
    'XGBoost': (pp_E_xgb, pb_E_xgb),
    'Random Forest': (pp_E_rf, pb_E_rf),
    'LightGBM': (pp_E_lgb, pb_E_lgb),
    'Improved DNN': (pp_E_dnn, pb_E_dnn),
    'Autoencoder+LR': (pp_E_ae, pb_E_ae)
}

plot_cm_all(preds_E, yte_E, LABEL_3,
            f"{PATH_E}/plots", "cm_all.png")

# ── XAI Versi E ──
print("\n🔍 XAI Versi E...")
df_imp_E, df_shap_E, corr_E, _, _ = run_all_xai(
    tn_E, Xtr_E, Xte_E, yte_E,
    sc_E, fc_E, LABEL_3,
    f"{PATH_E}/plots", prefix="E_", nc=3)

# ════════════════════════════════════════════════════════════════
# ╔═══════════════════════════════════════════════════╗
# ║  VERSI G: LOSO TEMPORAL + BINARY CLASSIFICATION   ║
# ╚═══════════════════════════════════════════════════╝
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("VERSI G — LOSO Temporal | Binary Classification")
print("="*60)

# ── LOSO split ──
train_rows_G, test_nisn_G = [], []
for nisn, grp in df_long.groupby('NISN'):
    grp = grp.sort_values('POSISI'); n = len(grp)
    if n < 3: continue
    if n >= 4: test_nisn_G.append(nisn)
    gt = grp.iloc[:-1]
    if len(gt) >= 2: train_rows_G.append((nisn, gt))

df_long_G_tr = pd.concat(
    [g for _,g in train_rows_G], ignore_index=True)
print(f"\n   Feature engineering TRAIN (LOSO)...")
df_ft_G_tr   = feature_engineering(df_long_G_tr, mapel_ada)
df_ft_G_te   = df_full[df_full['NISN'].isin(test_nisn_G)].copy()

# Label binary: Q75 dari train
srk_G_tr = df_ft_G_tr['skor_risiko_komposit']
Q75_G    = float(srk_G_tr.quantile(0.75))
print(f"   Q75_train (binary threshold) = {Q75_G:.4f}")

df_ft_G_tr['proxy_label'] = srk_G_tr.apply(
    lambda x: 0 if x<Q75_G else 1)
df_ft_G_te['proxy_label'] = df_ft_G_te[
    'skor_risiko_komposit'].apply(
    lambda x: 0 if x<Q75_G else 1)

fc_G = [c for c in FITUR_COLS
        if c in df_ft_G_tr.columns
        and c in df_ft_G_te.columns]

Xtr_G_r = df_ft_G_tr[fc_G].fillna(0).values.astype(np.float32)
Xte_G_r = df_ft_G_te[fc_G].fillna(0).values.astype(np.float32)
ytr_G_r = df_ft_G_tr['proxy_label'].values.astype(int)
yte_G   = df_ft_G_te['proxy_label'].values.astype(int)

sc_G = MinMaxScaler()
Xtr_G_s = sc_G.fit_transform(Xtr_G_r).astype(np.float32)
Xte_G   = sc_G.transform(Xte_G_r).astype(np.float32)
with open(f"{PATH_G}/models/scaler.pkl",'wb') as f:
    pickle.dump(sc_G, f)

mn_G = min((ytr_G_r==c).sum() for c in np.unique(ytr_G_r))
sm_G = SMOTE(random_state=SEED,k_neighbors=max(1,min(5,mn_G-1)))
Xtr_G, ytr_G = sm_G.fit_resample(Xtr_G_s, ytr_G_r)

print(f"\n   Train: {len(Xtr_G)} | Test: {len(Xte_G)}")
print(f"   Distribusi test: {np.bincount(yte_G,minlength=2)}")

# ── Optuna ──
print(f"\n🔍 Optuna Tuning Versi G (binary)...")
bp_G_tn,_  = run_optuna(make_optuna_tabnet(Xtr_G,ytr_G,2))
bp_G_xgb,_ = run_optuna(make_optuna_xgb(Xtr_G,ytr_G,2))
bp_G_rf,_  = run_optuna(make_optuna_rf(Xtr_G,ytr_G,2))
bp_G_lgb,_ = run_optuna(make_optuna_lgb(Xtr_G,ytr_G,2))
bp_G_all = {'TabNet':bp_G_tn,'XGBoost':bp_G_xgb,
             'RandomForest':bp_G_rf,'LightGBM':bp_G_lgb}
with open(f"{PATH_G}/best_params.json",'w') as f:
    json.dump(bp_G_all, f, indent=2)

# ── Training & evaluasi ──
print(f"\n🚀 Training Versi G...")

# TabNet
Xv1,Xv2,yv1,yv2 = train_test_split(
    Xtr_G,ytr_G,test_size=0.15,random_state=SEED,stratify=ytr_G)
tn_G = build_tabnet(bp_G_tn,2); tn_G.verbose=10
tn_G.fit(Xv1,yv1,eval_set=[(Xv2,yv2)],
         eval_metric=['accuracy'],max_epochs=200,patience=30,
         batch_size=64,virtual_batch_size=32,
         num_workers=0,drop_last=False)
tn_G.save_model(f"{PATH_G}/models/tabnet_model")
pp_G_tn  = tn_G.predict(Xte_G)
pb_G_tn  = tn_G.predict_proba(Xte_G)
m_G_tn   = hitung_metrik(yte_G,pp_G_tn,pb_G_tn,"TabNet",mode='binary')
print(classification_report(yte_G,pp_G_tn,
    target_names=list(LABEL_2.values()),zero_division=0))

# XGBoost
xp = {k:v for k,v in bp_G_xgb.items()}
xp.update({'use_label_encoder':False,'eval_metric':'logloss',
            'random_state':SEED,'n_jobs':-1})
xm_G = xgb.XGBClassifier(**xp); xm_G.fit(Xtr_G,ytr_G)
pp_G_xgb = xm_G.predict(Xte_G); pb_G_xgb = xm_G.predict_proba(Xte_G)
m_G_xgb  = hitung_metrik(yte_G,pp_G_xgb,pb_G_xgb,"XGBoost",mode='binary')
print(classification_report(yte_G,pp_G_xgb,
    target_names=list(LABEL_2.values()),zero_division=0))
with open(f"{PATH_G}/models/xgboost.pkl",'wb') as f:
    pickle.dump(xm_G,f)

# RF
rp = {k:v for k,v in bp_G_rf.items()}
rp.update({'class_weight':'balanced','random_state':SEED,'n_jobs':-1})
rm_G = RandomForestClassifier(**rp); rm_G.fit(Xtr_G,ytr_G)
pp_G_rf  = rm_G.predict(Xte_G); pb_G_rf = rm_G.predict_proba(Xte_G)
m_G_rf   = hitung_metrik(yte_G,pp_G_rf,pb_G_rf,"Random Forest",mode='binary')
print(classification_report(yte_G,pp_G_rf,
    target_names=list(LABEL_2.values()),zero_division=0))
with open(f"{PATH_G}/models/random_forest.pkl",'wb') as f:
    pickle.dump(rm_G,f)

# LightGBM
lp = {k:v for k,v in bp_G_lgb.items()}
lp.update({'class_weight':'balanced','random_state':SEED,'n_jobs':-1,'verbose':-1})
lm_G = lgb.LGBMClassifier(**lp); lm_G.fit(Xtr_G,ytr_G)
pp_G_lgb  = lm_G.predict(Xte_G); pb_G_lgb = lm_G.predict_proba(Xte_G)
m_G_lgb   = hitung_metrik(yte_G,pp_G_lgb,pb_G_lgb,"LightGBM",mode='binary')
print(classification_report(yte_G,pp_G_lgb,
    target_names=list(LABEL_2.values()),zero_division=0))
with open(f"{PATH_G}/models/lightgbm.pkl",'wb') as f:
    pickle.dump(lm_G,f)

# DNN
dm_G,pp_G_dnn,pb_G_dnn = train_dnn(Xtr_G,ytr_G,Xte_G,yte_G,nc=2)
m_G_dnn = hitung_metrik(yte_G,pp_G_dnn,pb_G_dnn,"Improved DNN",mode='binary')
print(classification_report(yte_G,pp_G_dnn,
    target_names=list(LABEL_2.values()),zero_division=0))
torch.save(dm_G.state_dict(),f"{PATH_G}/models/dnn.pt")

# AE+LR
ae_G,lr_G,pp_G_ae,pb_G_ae = train_ae_lr(Xtr_G,ytr_G,Xte_G,nc=2)
m_G_ae = hitung_metrik(yte_G,pp_G_ae,pb_G_ae,"Autoencoder+LR",mode='binary')
print(classification_report(yte_G,pp_G_ae,
    target_names=list(LABEL_2.values()),zero_division=0))
torch.save(ae_G.state_dict(),f"{PATH_G}/models/ae.pt")
with open(f"{PATH_G}/models/lr.pkl",'wb') as f:
    pickle.dump(lr_G,f)

# Ringkasan
semua_G = [m_G_tn,m_G_xgb,m_G_rf,m_G_lgb,m_G_dnn,m_G_ae]
df_G = pd.DataFrame(semua_G).set_index('Model').round(4)
df_G.to_csv(f"{PATH_G}/hasil_evaluasi.csv")
best_G = df_G['F1'].idxmax()

print(f"\n📊 VERSI G — HASIL (LOSO Binary)")
for nm,row in df_G.iterrows():
    mk = '★' if nm==best_G else ' '
    print(f"  {mk} {nm:<20} F1={row['F1']:.4f} AUC={row['AUC-ROC']:.4f}")

plot_bar_metrik(df_G, f"{PATH_G}/plots",
                "perbandingan_6model.png")
preds_G = {'TabNet':(pp_G_tn,pb_G_tn),
           'XGBoost':(pp_G_xgb,pb_G_xgb),
           'RF':(pp_G_rf,pb_G_rf),
           'LightGBM':(pp_G_lgb,pb_G_lgb),
           'DNN':(pp_G_dnn,pb_G_dnn),
           'AE+LR':(pp_G_ae,pb_G_ae)}
plot_cm_all(preds_G, yte_G, LABEL_2,
            f"{PATH_G}/plots", "cm_all.png")

# ROC Curve Versi G
fig, ax = plt.subplots(figsize=(9,7))
clrs_roc = ['#e74c3c','#3498db','#2ecc71',
             '#f39c12','#9b59b6','#1abc9c']
for (nm,(_,pb)),cl in zip(preds_G.items(),clrs_roc):
    try:
        fp,tp,_ = roc_curve(yte_G,pb[:,1])
        ax.plot(fp,tp,lw=2,color=cl,
                label=f'{nm} (AUC={auc(fp,tp):.3f})')
    except: pass
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Semua Model — Versi G (LOSO Binary)',
             fontsize=12,fontweight='bold')
ax.legend(fontsize=9,loc='lower right')
ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PATH_G}/plots/roc_all.png",dpi=300,bbox_inches='tight')
plt.close()

# ── XAI Versi G ──
print("\n🔍 XAI Versi G...")
df_imp_G, df_shap_G, corr_G, _, _ = run_all_xai(
    tn_G, Xtr_G, Xte_G, yte_G,
    sc_G, fc_G, LABEL_2,
    f"{PATH_G}/plots", prefix="G_", nc=2)

# ════════════════════════════════════════════════════════════════
# RINGKASAN LINTAS SKEMA + SIMPAN METADATA
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("📊 RINGKASAN FINAL — SEMUA VERSI")
print("="*60)

summary = {
    'Versi B (StratKFold 5-fold, 3 kelas)': {
        'TabNet_F1'   : f"{df_B_mean.loc['TabNet','F1']:.4f}"
                        f"±{df_B_std.loc['TabNet','F1']:.4f}",
        'TabNet_AUC'  : f"{df_B_mean.loc['TabNet','AUC-ROC']:.4f}"
                        f"±{df_B_std.loc['TabNet','AUC-ROC']:.4f}",
        'Best_model'  : best_B,
        'Best_F1'     : f"{df_B_mean.loc[best_B,'F1']:.4f}",
        'SHAP_corr'   : f"{corr_B:.4f}",
    },
    'Versi E (Temporal sem1-5/sem6)': {
        'TabNet_F1'   : f"{df_E.loc['TabNet','F1']:.4f}",
        'TabNet_AUC'  : f"{df_E.loc['TabNet','AUC-ROC']:.4f}",
        'Best_model'  : best_E,
        'Best_F1'     : f"{df_E.loc[best_E,'F1']:.4f}",
        'SHAP_corr'   : f"{corr_E:.4f}",
        'Kelas_hilang': [LABEL_3[c] for c in kls_hilang_E],
    },
    'Versi G (LOSO Binary)': {
        'TabNet_F1'   : f"{df_G.loc['TabNet','F1']:.4f}",
        'TabNet_AUC'  : f"{df_G.loc['TabNet','AUC-ROC']:.4f}",
        'Best_model'  : best_G,
        'Best_F1'     : f"{df_G.loc[best_G,'F1']:.4f}",
        'SHAP_corr'   : f"{corr_G:.4f}",
    },
}

with open(f"{PATH_V4}/ringkasan_semua_versi.json",
          'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# Tabel ringkasan lintas skema
print(f"\n  {'Versi':<30} {'TabNet F1':>12} "
      f"{'TabNet AUC':>12} {'Best Model':>15} "
      f"{'SHAP ρ':>8}")
print(f"  {'-'*80}")
for ver, info in summary.items():
    print(f"  {ver:<30} "
          f"{info['TabNet_F1']:>12} "
          f"{info['TabNet_AUC']:>12} "
          f"{info['Best_model']:>15} "
          f"{info['SHAP_corr']:>8}")

# ── Prediksi semua siswa dengan model final (TabNet Versi B) ──
print("\n📋 Prediksi semua siswa (model Versi B)...")
X_all_pred = sc_B_final.transform(
    df_full[FITUR_COLS].fillna(0).values.astype(np.float32))
y_pred_all  = tn_B_final.predict(X_all_pred)
y_proba_all = tn_B_final.predict_proba(X_all_pred)

df_pred = df_full[['NISN','NAMA','KELAS_TERAKHIR',
                   'N_SEMESTER','skor_risiko_komposit',
                   'label_3','label_2']].copy()
df_pred['prediksi_label']  = y_pred_all
df_pred['prediksi_nama']   = pd.Series(y_pred_all).map(LABEL_3)
for c in range(3):
    df_pred[f'prob_{c}'] = y_proba_all[:,c]
df_pred.to_csv(f"{PATH_V4}/prediksi_semua_siswa.csv",
               index=False)

# Metadata global
meta_global = {
    'versi': 'tesis_output_v4',
    'deskripsi': 'Framework Explainable TabNet v4',
    'path_output': PATH_V4,
    'skema': ['Versi B (StratGroupKFold)',
              'Versi E (Temporal)',
              'Versi G (LOSO Binary)'],
    'xai': ['Attention Masks','SHAP','LIME','Counterfactual'],
    'n_siswa': len(df_full),
    'n_fitur': len(FITUR_COLS),
    'fitur_cols': FITUR_COLS,
    'Q25_full': Q25_FULL, 'Q75_full': Q75_FULL,
    'ringkasan': summary,
    'leakage_check': {
        'scaler_fit_train_only': True,
        'smote_train_only': True,
        'optuna_val_from_train': True,
        'no_evalset_xtest': True,
        'no_sample_move': True,
        'test_eval_once': True,
    }
}
with open(f"{PATH_V4}/metadata_global.json",
          'w', encoding='utf-8') as f:
    json.dump(meta_global, f, ensure_ascii=False, indent=2)

print(f"\n{'='*60}")
print("✅ SELESAI — tesis_output_v4")
print(f"{'='*60}")
print(f"""
  Output tersimpan di: {PATH_V4}
  ├── dataset_fitur_full.csv
  ├── prediksi_semua_siswa.csv
  ├── ringkasan_semua_versi.json
  ├── metadata_global.json
  │
  ├── versi_B_stratgroupkfold/
  │   ├── hasil_mean.csv | hasil_std.csv
  │   ├── best_params.json
  │   ├── models/ (tabnet_final, scaler_final, ...)
  │   └── plots/  (attention, SHAP, LIME, CF, xai_summary,
  │               perbandingan_6model, cm_all)
  │
  ├── versi_E_temporal/
  │   ├── hasil_evaluasi.csv | best_params.json
  │   ├── models/ (tabnet, scaler, xgboost, rf, lgbm, ...)
  │   └── plots/  (attention, SHAP, LIME, CF, xai_summary,
  │               perbandingan_6model, cm_all)
  │
  ├── versi_G_loso_binary/
  │   ├── hasil_evaluasi.csv | best_params.json
  │   ├── models/ (tabnet, scaler, xgboost, rf, lgbm, ...)
  │   └── plots/  (attention, SHAP, LIME, CF, xai_summary,
  │               perbandingan_6model, cm_all, roc_all)
  │
  └── xai_combined/

  Selanjutnya: jalankan dashboard → python dashboard_app.py
""")

# ════════════════════════════════════════════════════════════════
# PROTOTIPE DASHBOARD — SIMPAN KE DRIVE
# Jalankan terpisah di terminal: streamlit run dashboard_app.py
# ════════════════════════════════════════════════════════════════

DASHBOARD_CODE = '''# dashboard_app.py
# Prototipe Dashboard Explainable TabNet
# Prediksi Risiko Akademik Siswa SMP
# Jalankan: streamlit run dashboard_app.py
# ─────────────────────────────────────────

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import pickle, json, os

# ── Konfigurasi halaman ──────────────────────────────────────────
st.set_page_config(
    page_title="Dashboard Risiko Akademik Siswa SMP",
    page_icon="🏫",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── CSS kustom ───────────────────────────────────────────────────
st.markdown("""
<style>
  .main-header {
    background: linear-gradient(90deg,#1a3a5c,#2c7bb6);
    padding: 1.2rem 1.5rem;
    border-radius: 10px;
    color: white;
    margin-bottom: 1.5rem;
  }
  .main-header h1 { color:white; font-size:1.6rem; margin:0; }
  .main-header p  { color:#cfe2f3; font-size:0.9rem; margin:0.3rem 0 0 0; }
  .card-risiko-tinggi {
    background:#fff5f5; border:2px solid #e53e3e;
    border-radius:10px; padding:1rem;
  }
  .card-risiko-sedang {
    background:#fffaf0; border:2px solid #dd6b20;
    border-radius:10px; padding:1rem;
  }
  .card-aman {
    background:#f0fff4; border:2px solid #38a169;
    border-radius:10px; padding:1rem;
  }
  .badge-tinggi {
    background:#e53e3e; color:white;
    padding:4px 12px; border-radius:20px;
    font-weight:bold; font-size:0.85rem;
  }
  .badge-sedang {
    background:#dd6b20; color:white;
    padding:4px 12px; border-radius:20px;
    font-weight:bold; font-size:0.85rem;
  }
  .badge-rendah {
    background:#38a169; color:white;
    padding:4px 12px; border-radius:20px;
    font-weight:bold; font-size:0.85rem;
  }
  .badge-aman {
    background:#38a169; color:white;
    padding:4px 12px; border-radius:20px;
    font-weight:bold; font-size:0.85rem;
  }
  .metric-card {
    background:#f8fafc; border:1px solid #e2e8f0;
    border-radius:8px; padding:0.8rem; text-align:center;
  }
  .rekomendasi-box {
    background:#ebf8ff; border-left:4px solid #2c7bb6;
    padding:0.8rem 1rem; border-radius:0 8px 8px 0;
    margin:0.5rem 0;
  }
  .warning-box {
    background:#fffbeb; border-left:4px solid #d69e2e;
    padding:0.8rem 1rem; border-radius:0 8px 8px 0;
    margin:0.5rem 0;
  }
</style>
""", unsafe_allow_html=True)

# ── Header ───────────────────────────────────────────────────────
st.markdown("""
<div class="main-header">
  <h1>🏫 Dashboard Prediksi Risiko Akademik Siswa SMP</h1>
  <p>Framework Explainable TabNet | SMPN 1 Cibadak, Sukabumi |
     Powered by TabNet + SHAP + LIME + Counterfactual</p>
</div>
""", unsafe_allow_html=True)

# ── Path (sesuaikan dengan lokasi file) ──────────────────────────
BASE_PATH = "/content/drive/MyDrive/tesis_output_v4"
PATH_B    = f"{BASE_PATH}/versi_B_stratgroupkfold"

@st.cache_data
def load_data():
    df = pd.read_csv(f"{BASE_PATH}/prediksi_semua_siswa.csv")
    return df

@st.cache_resource
def load_fi():
    try:
        df = pd.read_csv(
            f"{PATH_B}/plots/B_attention_importance.csv")
        return df
    except: return None

@st.cache_data
def load_shap_vs_att():
    try:
        df = pd.read_csv(
            f"{PATH_B}/plots/B_shap_vs_attention.csv")
        return df
    except: return None

@st.cache_data
def load_lime():
    try:
        with open(f"{PATH_B}/plots/B_lime_results.json",
                  "r", encoding="utf-8") as f:
            return json.load(f)
    except: return []

@st.cache_data
def load_cf():
    try:
        with open(f"{PATH_B}/plots/B_counterfactual.json",
                  "r", encoding="utf-8") as f:
            return json.load(f)
    except: return []

df_pred = load_data()
df_fi   = load_fi()
lime_res = load_lime()
cf_res   = load_cf()
df_shap_cmp = load_shap_vs_att()

# ── Mapping label ─────────────────────────────────────────────────
LABEL_WARNA = {
    "Risiko Rendah" : ("badge-rendah",  "🟢"),
    "Risiko Sedang" : ("badge-sedang",  "🟡"),
    "Risiko Tinggi" : ("badge-tinggi",  "🔴"),
    "Tidak Berisiko": ("badge-aman",    "🟢"),
    "Berisiko"      : ("badge-tinggi",  "🔴"),
}

# ════════════════════════════════════════════════════════════════
# SIDEBAR
# ════════════════════════════════════════════════════════════════

with st.sidebar:
    st.image("https://via.placeholder.com/150x60/1a3a5c/white?"
             "text=TabNet+XAI", width=200)
    st.markdown("---")
    st.markdown("### 🗂️ Navigasi")
    halaman = st.radio("", [
        "🏠 Beranda",
        "📋 Daftar Siswa",
        "🔍 Profil Siswa",
        "📊 Explainability",
        "💡 Rekomendasi",
        "📈 Evaluasi Model",
    ])
    st.markdown("---")
    st.markdown("### 🎚️ Filter Kelas")
    kelas_list = ["Semua"] + sorted(
        df_pred["KELAS_TERAKHIR"].dropna().unique().tolist())
    filter_kelas = st.selectbox("Kelas", kelas_list)
    st.markdown("---")
    st.caption("Framework: Explainable TabNet v4\n"
               "SMPN 1 Cibadak © 2025")

# ════════════════════════════════════════════════════════════════
# HALAMAN: BERANDA
# ════════════════════════════════════════════════════════════════

if "Beranda" in halaman:
    # Statistik ringkasan
    total = len(df_pred)
    col1,col2,col3,col4 = st.columns(4)
    pred_col = ("prediksi_nama" if "prediksi_nama" in df_pred.columns
                else "label_nama")

    with col1:
        st.metric("Total Siswa", total)
    with col2:
        n_tinggi = (df_pred[pred_col]
                    .isin(["Risiko Tinggi","Berisiko"]).sum())
        pct_t = n_tinggi/total*100
        st.metric("🔴 Berisiko / Tinggi",
                  f"{n_tinggi} ({pct_t:.1f}%)")
    with col3:
        n_sedang = (df_pred[pred_col]
                    .isin(["Risiko Sedang"]).sum())
        st.metric("🟡 Risiko Sedang",
                  f"{n_sedang} ({n_sedang/total*100:.1f}%)")
    with col4:
        n_rendah = (df_pred[pred_col]
                    .isin(["Risiko Rendah","Tidak Berisiko"]).sum())
        st.metric("🟢 Aman / Rendah",
                  f"{n_rendah} ({n_rendah/total*100:.1f}%)")

    st.markdown("---")
    col_a, col_b = st.columns([1,1])

    with col_a:
        st.subheader("Distribusi Prediksi Risiko")
        dist = df_pred[pred_col].value_counts()
        fig, ax = plt.subplots(figsize=(6,5))
        colors = ["#e53e3e","#dd6b20","#38a169","#2c7bb6"]
        wedges, texts, autotexts = ax.pie(
            dist.values,
            labels=dist.index,
            autopct="%1.1f%%",
            colors=colors[:len(dist)],
            startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":2})
        for t in autotexts:
            t.set_fontsize(10); t.set_fontweight("bold")
        ax.axis("equal")
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with col_b:
        st.subheader("Distribusi per Kelas")
        if "KELAS_TERAKHIR" in df_pred.columns:
            cross = pd.crosstab(
                df_pred["KELAS_TERAKHIR"],
                df_pred[pred_col])
            fig2, ax2 = plt.subplots(figsize=(8,5))
            cross.plot(kind="bar", ax=ax2,
                       color=colors[:len(cross.columns)],
                       edgecolor="white", alpha=0.85)
            ax2.set_xlabel("Kelas", fontsize=10)
            ax2.set_ylabel("Jumlah Siswa", fontsize=10)
            ax2.legend(fontsize=8, loc="upper right")
            ax2.tick_params(axis="x", rotation=0)
            ax2.grid(True, alpha=0.3, axis="y")
            st.pyplot(fig2, use_container_width=True)
            plt.close()

    # Alert siswa berisiko tinggi
    df_risiko = df_pred[
        df_pred[pred_col].isin(
            ["Risiko Tinggi","Berisiko"])]
    if len(df_risiko) > 0:
        st.markdown("---")
        st.subheader(f"⚠️ Siswa Memerlukan Perhatian Segera "
                     f"({len(df_risiko)} siswa)")
        prob_col = ("prob_2" if "prob_2" in df_pred.columns
                    else "prob_1")
        show_cols = ["NAMA","KELAS_TERAKHIR",
                     pred_col, prob_col]
        show_cols = [c for c in show_cols if c in df_risiko.columns]
        st.dataframe(
            df_risiko[show_cols].rename(columns={
                pred_col: "Status Risiko",
                prob_col: "Probabilitas"}
            ).sort_values("Probabilitas", ascending=False)
            .head(10).reset_index(drop=True),
            use_container_width=True)

# ════════════════════════════════════════════════════════════════
# HALAMAN: DAFTAR SISWA
# ════════════════════════════════════════════════════════════════

elif "Daftar Siswa" in halaman:
    st.subheader("📋 Daftar Seluruh Siswa")
    pred_col = ("prediksi_nama" if "prediksi_nama" in df_pred.columns
                else "label_nama")

    df_show = df_pred.copy()
    if filter_kelas != "Semua":
        df_show = df_show[
            df_show["KELAS_TERAKHIR"] == filter_kelas]

    col_f1, col_f2 = st.columns([2,1])
    with col_f1:
        cari = st.text_input("🔎 Cari nama siswa...")
    with col_f2:
        filter_risiko = st.selectbox(
            "Filter Status",
            ["Semua","Berisiko/Tinggi","Sedang","Aman/Rendah"])

    if cari:
        df_show = df_show[
            df_show["NAMA"].str.contains(
                cari, case=False, na=False)]
    if filter_risiko == "Berisiko/Tinggi":
        df_show = df_show[
            df_show[pred_col].isin(
                ["Risiko Tinggi","Berisiko"])]
    elif filter_risiko == "Sedang":
        df_show = df_show[
            df_show[pred_col] == "Risiko Sedang"]
    elif filter_risiko == "Aman/Rendah":
        df_show = df_show[
            df_show[pred_col].isin(
                ["Risiko Rendah","Tidak Berisiko"])]

    tampil_cols = ["NAMA","KELAS_TERAKHIR","N_SEMESTER",
                   pred_col,"skor_risiko_komposit"]
    tampil_cols = [c for c in tampil_cols
                   if c in df_show.columns]

    st.dataframe(
        df_show[tampil_cols].rename(columns={
            pred_col: "Status Risiko",
            "skor_risiko_komposit": "SRK"}
        ).reset_index(drop=True),
        use_container_width=True, height=450)
    st.caption(f"Menampilkan {len(df_show)} dari "
               f"{len(df_pred)} siswa")

    # Download
    csv_dl = df_show[tampil_cols].to_csv(index=False)
    st.download_button(
        "⬇️ Unduh Data CSV",
        data=csv_dl,
        file_name="daftar_risiko_siswa.csv",
        mime="text/csv")

# ════════════════════════════════════════════════════════════════
# HALAMAN: PROFIL SISWA
# ════════════════════════════════════════════════════════════════

elif "Profil Siswa" in halaman:
    st.subheader("🔍 Profil Individual Siswa")
    pred_col = ("prediksi_nama" if "prediksi_nama" in df_pred.columns
                else "label_nama")

    df_filter = df_pred.copy()
    if filter_kelas != "Semua":
        df_filter = df_filter[
            df_filter["KELAS_TERAKHIR"] == filter_kelas]

    nama_list = sorted(df_filter["NAMA"].dropna().unique())
    siswa_dipilih = st.selectbox(
        "Pilih Siswa:", nama_list)

    if siswa_dipilih:
        row = df_filter[
            df_filter["NAMA"] == siswa_dipilih].iloc[0]
        status = row.get(pred_col, "-")
        badge, ikon = LABEL_WARNA.get(
            status, ("badge-sedang","⚪"))

        # Header profil
        col_p1, col_p2 = st.columns([1,2])
        with col_p1:
            st.markdown(f"""
            <div class="metric-card">
              <h2 style="font-size:3rem;margin:0">{ikon}</h2>
              <span class="{badge}">{status}</span>
            </div>
            """, unsafe_allow_html=True)
        with col_p2:
            st.markdown(f"**Nama:** {row['NAMA']}")
            st.markdown(f"**Kelas:** "
                        f"{row.get('KELAS_TERAKHIR','-')}")
            st.markdown(f"**Semester data:** "
                        f"{row.get('N_SEMESTER','-')}")
            srk = row.get("skor_risiko_komposit","-")
            st.markdown(f"**Skor Risiko (SRK):** "
                        f"`{srk}`")

        st.markdown("---")
        # Metrik utama
        st.subheader("Indikator Risiko Akademik")
        col_m1,col_m2,col_m3 = st.columns(3)
        with col_m1:
            v = row.get("rata_nilai_inti_semua","-")
            delta_c = ("normal" if isinstance(v,float)
                       and v >= 75 else "inverse")
            st.metric("📚 Rata-rata Nilai",
                      f"{v:.1f}" if isinstance(v,float) else v)
        with col_m2:
            v2 = row.get("persentase_kehadiran_rata","-")
            st.metric("📅 Kehadiran (%)",
                      f"{v2:.1f}%" if isinstance(v2,float) else v2)
        with col_m3:
            v3 = row.get("total_alfa_6semester","-")
            st.metric("⚠️ Total Alfa",
                      f"{int(v3)}" if isinstance(v3,(int,float))
                      else v3)

        col_m4, col_m5, col_m6 = st.columns(3)
        with col_m4:
            v4 = row.get("rata_nilai_ekskul","-")
            st.metric("🎯 Rata Ekskul",
                      f"{v4:.2f}" if isinstance(v4,float) else v4)
        with col_m5:
            v5 = row.get("tren_nilai","-")
            st.metric("📈 Tren Nilai",
                      f"{v5:.3f}" if isinstance(v5,float) else v5)
        with col_m6:
            v6 = row.get("total_ketidakhadiran","-")
            st.metric("🏃 Total Tidak Hadir",
                      f"{int(v6)}" if isinstance(v6,(int,float))
                      else v6)

        # Probabilitas prediksi
        prob_cols = [c for c in df_pred.columns
                     if c.startswith("prob_")]
        if prob_cols:
            st.markdown("---")
            st.subheader("Probabilitas Prediksi Model")
            prob_vals = [row.get(c,0) for c in prob_cols]
            label_map = {0:"Rendah/Aman",1:"Sedang",2:"Tinggi/Risiko"}
            fig_p, ax_p = plt.subplots(figsize=(7,3))
            bars_p = ax_p.barh(
                [label_map.get(i,f"Kelas {i}")
                 for i in range(len(prob_vals))],
                prob_vals,
                color=["#38a169","#dd6b20","#e53e3e"][
                    :len(prob_vals)],
                alpha=0.85)
            for bar,val in zip(bars_p,prob_vals):
                ax_p.text(val+0.01,
                          bar.get_y()+bar.get_height()/2,
                          f"{val:.3f}",va="center",fontsize=9)
            ax_p.set_xlim(0,1.1)
            ax_p.set_xlabel("Probabilitas")
            ax_p.grid(True,alpha=0.3,axis="x")
            st.pyplot(fig_p, use_container_width=True)
            plt.close()

# ════════════════════════════════════════════════════════════════
# HALAMAN: EXPLAINABILITY
# ════════════════════════════════════════════════════════════════

elif "Explainability" in halaman:
    st.subheader("📊 Explainability — 4 Metode XAI")

    tab1,tab2,tab3,tab4 = st.tabs([
        "🔵 Attention Masks",
        "🔴 SHAP",
        "🟡 LIME",
        "🟢 Counterfactual"])

    # Tab 1: Attention Masks
    with tab1:
        st.markdown("#### Global Feature Importance (Attention Masks)")
        st.markdown(
            "Attention masks TabNet menunjukkan kontribusi setiap "
            "fitur terhadap prediksi secara global. "
            "Ini adalah penjelasan inheren model — tidak memerlukan "
            "metode post-hoc tambahan.")
        if df_fi is not None:
            fig_att, ax_att = plt.subplots(figsize=(10,7))
            t15 = df_fi.head(15)
            bars_a = ax_att.barh(
                t15["Fitur"][::-1],
                t15["Importance"][::-1],
                color="#2c7bb6", alpha=0.85)
            for bar,val in zip(bars_a,
                               t15["Importance"][::-1].values):
                ax_att.text(val+0.002,
                            bar.get_y()+bar.get_height()/2,
                            f"{val:.4f}",va="center",fontsize=8)
            ax_att.set_xlabel("Skor Kepentingan Fitur",fontsize=10)
            ax_att.set_ylabel("Fitur",fontsize=10)
            ax_att.grid(True,alpha=0.3,axis="x")
            st.pyplot(fig_att, use_container_width=True)
            plt.close()
            st.dataframe(df_fi.head(10), use_container_width=True)
        else:
            st.info("File attention_importance.csv belum tersedia. "
                    "Jalankan pipeline training terlebih dahulu.")

        # Gambar dari file
        att_img = f"{PATH_B}/plots/B_attention_masks.png"
        if os.path.exists(att_img):
            st.image(att_img,
                     caption="Attention Masks TabNet (Top-15 Fitur)",
                     use_column_width=True)

    # Tab 2: SHAP
    with tab2:
        st.markdown("#### SHAP — Validasi Model-Agnostic")
        st.markdown(
            "SHAP (SHapley Additive exPlanations) memvalidasi "
            "attention masks dengan pendekatan yang independen "
            "dari arsitektur model. Spearman rank correlation "
            "antara SHAP dan attention digunakan sebagai "
            "ukuran validitas.")
        if df_shap_cmp is not None:
            corr_val = df_shap_cmp[[
                "rank_shap","rank_att"]].corr(
                method="spearman").iloc[0,1]
            if corr_val >= 0.7:
                st.success(f"✅ Spearman ρ = {corr_val:.3f} — "
                           f"Attention masks VALID dan konsisten "
                           f"dengan SHAP")
            else:
                st.warning(f"⚠️ Spearman ρ = {corr_val:.3f} — "
                           f"Ada perbedaan antara SHAP dan attention")
            st.dataframe(df_shap_cmp.head(10),
                         use_container_width=True)

        # Gambar dari file
        shap_img = f"{PATH_B}/plots/B_shap_vs_attention.png"
        if os.path.exists(shap_img):
            st.image(shap_img,
                     caption="(a) SHAP Global vs (b) Attention Masks",
                     use_column_width=True)
        shap_sum = f"{PATH_B}/plots/B_shap_summary.png"
        if os.path.exists(shap_sum):
            st.image(shap_sum,
                     caption="SHAP Summary Plot (Beeswarm)",
                     use_column_width=True)

    # Tab 3: LIME
    with tab3:
        st.markdown("#### LIME — Penjelasan Per Siswa")
        st.markdown(
            "LIME memberikan penjelasan lokal untuk setiap siswa. "
            "Batang **merah** = fitur yang meningkatkan risiko, "
            "batang **hijau** = fitur protektif. "
            "Rata-rata dari 3 kali eksekusi untuk stabilitas.")

        if lime_res:
            siswa_lime = [f"Siswa idx={r['idx']} | "
                          f"{r['pred_nama']}"
                          for r in lime_res]
            sel_lime = st.selectbox(
                "Pilih siswa:", siswa_lime)
            idx_l = siswa_lime.index(sel_lime)
            r = lime_res[idx_l]

            fig_l, ax_l = plt.subplots(figsize=(9,5))
            lv = r["lime_vals"][:8]
            fs = [v[0][:35] for v in lv]
            sc = [v[1] for v in lv]
            cb = ["#d7191c" if s>0 else "#1a9641" for s in sc]
            ax_l.barh(fs[::-1], sc[::-1],
                      color=cb[::-1], alpha=0.85)
            ax_l.axvline(0, color="black", lw=0.8, ls="--")
            ax_l.set_xlabel("Skor LIME (rata-rata 3 run)",
                            fontsize=9)
            ax_l.grid(True,alpha=0.3,axis="x")
            st.pyplot(fig_l, use_container_width=True)
            plt.close()

            col_lime1, col_lime2 = st.columns(2)
            with col_lime1:
                st.metric("Prediksi", r["pred_nama"])
            with col_lime2:
                st.metric("Probabilitas",
                          f"{r['proba'][r['pred_label']]:.3f}")
        else:
            lime_img = f"{PATH_B}/plots/B_lime_panel.png"
            if os.path.exists(lime_img):
                st.image(lime_img,
                         caption="LIME Explanation Panel",
                         use_column_width=True)
            else:
                st.info("Data LIME belum tersedia.")

    # Tab 4: Counterfactual
    with tab4:
        st.markdown("#### Counterfactual — Rekomendasi What-If")
        st.markdown(
            "Counterfactual menunjukkan **apa yang perlu diubah** "
            "pada profil siswa agar statusnya berubah dari "
            "Berisiko menjadi Tidak Berisiko.")

        if cf_res:
            siswa_cf = [f"Siswa idx={r['indeks']} | "
                        f"{r['pred_asal_nm']} → "
                        f"{r['pred_akhir_nm']} "
                        f"{'✅' if r['berhasil'] else '⚠️'}"
                        for r in cf_res]
            sel_cf = st.selectbox("Pilih siswa:", siswa_cf)
            idx_c  = siswa_cf.index(sel_cf)
            rc     = cf_res[idx_c]

            if rc["berhasil"]:
                st.success(
                    f"✅ Dengan perubahan berikut, prediksi "
                    f"berubah dari **{rc['pred_asal_nm']}** "
                    f"→ **{rc['pred_akhir_nm']}**")
            else:
                st.warning(
                    f"⚠️ Rekomendasi berikut belum cukup "
                    f"mengubah prediksi secara penuh.")

            st.markdown("**Langkah rekomendasi guru:**")
            for i, s in enumerate(rc["skenario"][:5], 1):
                warna = ("#ebf8ff" if s["arah"]=="naikkan"
                         else "#fff5f5")
                icon  = "⬆️" if s["arah"]=="naikkan" else "⬇️"
                st.markdown(f"""
                <div style="background:{warna};
                  border-radius:8px; padding:0.6rem 1rem;
                  margin:0.3rem 0; border-left:3px solid
                  {'#2c7bb6' if s['arah']=='naikkan' else '#e53e3e'};">
                  <strong>{i}. {icon} {s['rekomendasi']}</strong><br/>
                  <small>Dari <code>{s['nilai_asal']}</code>
                  → Target <code>{s['nilai_baru']}</code> |
                  Prediksi baru: <strong>{s['pred_nama']}</strong>
                  </small>
                </div>
                """, unsafe_allow_html=True)
        else:
            cf_img = f"{PATH_B}/plots/B_counterfactual_table.png"
            if os.path.exists(cf_img):
                st.image(cf_img,
                         caption="Tabel Rekomendasi Counterfactual",
                         use_column_width=True)
            else:
                st.info("Data counterfactual belum tersedia.")

    # Gambar ringkasan XAI 4-panel
    xai_sum_img = f"{PATH_B}/plots/B_xai_summary_4panel.png"
    if os.path.exists(xai_sum_img):
        st.markdown("---")
        st.subheader("Ringkasan XAI — 4 Metode")
        st.image(xai_sum_img,
                 caption="(a) Attention (b) SHAP "
                          "(c) LIME (d) Status Counterfactual",
                 use_column_width=True)

# ════════════════════════════════════════════════════════════════
# HALAMAN: REKOMENDASI
# ════════════════════════════════════════════════════════════════

elif "Rekomendasi" in halaman:
    st.subheader("💡 Rekomendasi Intervensi Guru")
    pred_col = ("prediksi_nama" if "prediksi_nama" in df_pred.columns
                else "label_nama")

    df_risiko = df_pred[
        df_pred[pred_col].isin(["Risiko Tinggi","Berisiko"])
    ].copy()

    if filter_kelas != "Semua":
        df_risiko = df_risiko[
            df_risiko["KELAS_TERAKHIR"] == filter_kelas]

    st.markdown(f"### Siswa Berisiko: **{len(df_risiko)}** siswa")

    REKOMENDASI_UMUM = {
        "total_alfa_6semester":
            "Pantau kehadiran siswa dan komunikasikan dengan orang tua",
        "persentase_kehadiran_rata":
            "Tingkatkan monitoring kehadiran harian",
        "rata_nilai_ekskul":
            "Dorong siswa untuk aktif di ekstrakurikuler",
        "konsistensi_ekskul":
            "Pastikan siswa konsisten mengikuti ekstrakurikuler",
        "rata_nilai_inti_semua":
            "Berikan bimbingan belajar tambahan",
        "jumlah_mapel_dibawah_75":
            "Identifikasi mata pelajaran yang perlu remedial",
        "tren_nilai":
            "Cermati penurunan nilai dan beri motivasi belajar",
        "total_ketidakhadiran":
            "Lakukan home visit jika ketidakhadiran tinggi",
    }

    for _, row in df_risiko.head(5).iterrows():
        nama   = row.get("NAMA","-")
        kelas  = row.get("KELAS_TERAKHIR","-")
        status = row.get(pred_col,"-")
        srk    = row.get("skor_risiko_komposit",0)
        badge, ikon = LABEL_WARNA.get(
            status,("badge-sedang","⚪"))

        with st.expander(
                f"{ikon} {nama} — Kelas {kelas} | "
                f"SRK: {srk:.3f}"):
            col_r1, col_r2 = st.columns([1,2])
            with col_r1:
                st.markdown(
                    f"**Status:** "
                    f'<span class="{badge}">{status}</span>',
                    unsafe_allow_html=True)
                st.metric("Nilai Rata",
                          f"{row.get('rata_nilai_inti_semua',0):.1f}")
                st.metric("Kehadiran",
                          f"{row.get('persentase_kehadiran_rata',0):.1f}%")
                st.metric("Total Alfa",
                          f"{int(row.get('total_alfa_6semester',0))}")
            with col_r2:
                st.markdown("**Rekomendasi Tindakan:**")
                faktor_risiko = []
                if row.get("persentase_kehadiran_rata",100) < 80:
                    faktor_risiko.append("persentase_kehadiran_rata")
                if row.get("total_alfa_6semester",0) > 10:
                    faktor_risiko.append("total_alfa_6semester")
                if row.get("rata_nilai_inti_semua",100) < 70:
                    faktor_risiko.append("rata_nilai_inti_semua")
                if row.get("rata_nilai_ekskul",4) < 2:
                    faktor_risiko.append("rata_nilai_ekskul")
                if not faktor_risiko:
                    faktor_risiko = list(REKOMENDASI_UMUM.keys())[:3]

                for fi_ in faktor_risiko[:4]:
                    rek = REKOMENDASI_UMUM.get(
                        fi_,"Pantau perkembangan siswa")
                    st.markdown(
                        f'<div class="rekomendasi-box">'
                        f"📌 {rek}</div>",
                        unsafe_allow_html=True)

    st.markdown("---")
    st.info("💡 **Catatan untuk Guru:** Rekomendasi ini dihasilkan "
            "secara otomatis berdasarkan model AI dan perlu "
            "divalidasi dengan penilaian profesional guru "
            "di lapangan.")

# ════════════════════════════════════════════════════════════════
# HALAMAN: EVALUASI MODEL
# ════════════════════════════════════════════════════════════════

elif "Evaluasi Model" in halaman:
    st.subheader("📈 Evaluasi Performa 6 Model")
    st.markdown(
        "Perbandingan performa TabNet dengan 5 model baseline "
        "menggunakan tiga skema evaluasi komplementer.")

    tab_b, tab_e, tab_g = st.tabs([
        "Versi B (StratKFold)",
        "Versi E (Temporal)",
        "Versi G (LOSO Binary)"])

    def tampil_hasil(csv_path, std_path=None, label=""):
        if os.path.exists(csv_path):
            df_h = pd.read_csv(csv_path, index_col=0)
            if std_path and os.path.exists(std_path):
                df_s = pd.read_csv(std_path, index_col=0)
                st.subheader(f"Hasil {label} (Mean ± Std)")
                disp = pd.DataFrame()
                for col in df_h.columns:
                    if col in df_s.columns:
                        disp[col] = (
                            df_h[col].round(4).astype(str)
                            + " ±"
                            + df_s[col].round(4).astype(str))
                    else:
                        disp[col] = df_h[col].round(4)
                st.dataframe(disp, use_container_width=True)
            else:
                st.dataframe(df_h.round(4),
                             use_container_width=True)

            # Bar chart F1
            if "F1" in df_h.columns:
                fig_e, ax_e = plt.subplots(figsize=(8,4))
                f1_vals = df_h["F1"].values
                colors  = ["#e53e3e" if n=="TabNet"
                           else "#2c7bb6"
                           for n in df_h.index]
                bars_e  = ax_e.bar(df_h.index, f1_vals,
                                    color=colors, alpha=0.85)
                for bar,val in zip(bars_e,f1_vals):
                    ax_e.text(bar.get_x()+bar.get_width()/2,
                              val+0.005, f"{val:.4f}",
                              ha="center",fontsize=8,
                              fontweight="bold")
                ax_e.set_ylabel("F1-macro/binary")
                ax_e.set_ylim(0, min(1.1, max(f1_vals)*1.2+0.05))
                ax_e.tick_params(axis="x",rotation=15)
                ax_e.grid(True,alpha=0.3,axis="y")
                st.pyplot(fig_e, use_container_width=True)
                plt.close()
        else:
            st.info(f"File {csv_path} belum tersedia.")

    with tab_b:
        tampil_hasil(
            f"{PATH_B}/hasil_mean.csv",
            f"{PATH_B}/hasil_std.csv",
            "Versi B (StratKFold 5-fold)")
        img_b = f"{PATH_B}/plots/perbandingan_6model.png"
        if os.path.exists(img_b):
            st.image(img_b, use_column_width=True)

    with tab_e:
        tampil_hasil(
            f"{BASE_PATH}/versi_E_temporal/hasil_evaluasi.csv",
            label="Versi E (Temporal Split)")
        img_e = (f"{BASE_PATH}/versi_E_temporal/plots/"
                 "perbandingan_6model.png")
        if os.path.exists(img_e):
            st.image(img_e, use_column_width=True)

    with tab_g:
        tampil_hasil(
            f"{BASE_PATH}/versi_G_loso_binary/hasil_evaluasi.csv",
            label="Versi G (LOSO Binary)")
        img_g = (f"{BASE_PATH}/versi_G_loso_binary/plots/"
                 "perbandingan_6model.png")
        if os.path.exists(img_g):
            st.image(img_g, use_column_width=True)
        roc_g = (f"{BASE_PATH}/versi_G_loso_binary/plots/"
                 "roc_all.png")
        if os.path.exists(roc_g):
            st.image(roc_g,
                     caption="ROC Curve Semua Model (Versi G)",
                     use_column_width=True)

    st.markdown("---")
    st.markdown("""
    **Keterangan Skema Evaluasi:**
    - **Versi B (StratGroupKFold):** Evaluasi utama, 5-fold,
      3 kelas, mean±std, group=NISN
    - **Versi E (Temporal):** Simulasi nyata, sem 1-5 train,
      sem 6 test
    - **Versi G (LOSO Binary):** Per-siswa temporal,
      binary Berisiko/Tidak
    """)
'''

# Simpan dashboard ke Drive
dashboard_path = f"{PATH_V4}/dashboard_app.py"
with open(dashboard_path, 'w', encoding='utf-8') as f:
    f.write(DASHBOARD_CODE)

print(f"\n✅ Dashboard tersimpan → {dashboard_path}")
print(f"   Jalankan di terminal: streamlit run {dashboard_path}")
print(f"   Atau di Colab cell baru:")
print(f"   !streamlit run {dashboard_path} &")
print(f"   dari ngrok: !npx localtunnel --port 8501")

✅ Library siap | Device: cpu
   Framework: Explainable TabNet v4
   Skema: Versi B (StratKFold) + E (Temporal) + G (LOSO Binary)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Path output: /content/drive/MyDrive/tesis_output_v4

📂 Load data mentah...
   Baris     : 5509
   Siswa     : 1248
   Mapel     : ['PAI', 'PKN', 'B_INDO', 'B_ING', 'MTK', 'IPA', 'IPS', 'B_SUN', 'PJOK', 'PKRY', 'SENI']
   Posisi    : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
   Sem akhir : 6

⚙️  Feature engineering — populasi penuh...
   ✅ 1248 siswa × 27 kolom

   Fitur model: 21 (N_SEMESTER tidak termasuk)
   Q25 (populasi): 2.0566
   Q75 (populasi): 3.3677
   Unique SRK   : 1206 dari 1248

   Dataset lengkap → /content/drive/MyDrive/tesis_output_v4/dataset_fitur_full.csv

✅ Semua fungsi siap (XAI sama rata, 300dpi, label (a)(b)...)

VERSI B — StratifiedGroupKFold 5-fold | 3 Kelas

   Siswa: 

  0%|          | 0/50 [00:00<?, ?it/s]

   Spearman ρ=0.8234 (p=0.0000) → VALID
   ✅ [XAI-2] SHAP summary beeswarm tersimpan
   ✅ [XAI-2] SHAP selesai (ρ=0.823)

  [3/4] LIME...
   ✅ [XAI-3] LIME (3 sampel) → B_lime_panel.png

  [4/4] Counterfactual...
   CF idx=33: Risiko Tinggi → Risiko Sedang ✅
   CF idx=36: Risiko Tinggi → Risiko Tinggi ⚠️
   CF idx=46: Risiko Tinggi → Risiko Sedang ✅
   CF idx=0: Risiko Sedang → Risiko Rendah ✅
   CF idx=2: Risiko Sedang → Risiko Rendah ✅
   CF idx=4: Risiko Sedang → Risiko Rendah ✅
   Validasi CF-SHAP: 3/6 konsisten (50%)
   ✅ [XAI-4] Counterfactual tabel → B_counterfactual_table.png
   ✅ Ringkasan XAI 4-panel → B_xai_summary_4panel.png

  ✅ Semua XAI selesai (B_)

   Training TabNet final (semua data) untuk dashboard...

Early stopping occurred at epoch 61 with best_epoch = 36 and best_val_0_accuracy = 0.98932
Successfully saved model at /content/drive/MyDrive/tesis_output_v4/versi_B_stratgroupkfold/models/tabnet_final.zip
   ✅ Model final Versi B tersimpan

VERSI E — Temporal Split |

  0%|          | 0/50 [00:00<?, ?it/s]

   Spearman ρ=0.8688 (p=0.0000) → VALID
   ✅ [XAI-2] SHAP summary beeswarm tersimpan
   ✅ [XAI-2] SHAP selesai (ρ=0.869)

  [3/4] LIME...
   ⚠️ Skip LIME kelas Risiko Rendah: tidak ada
   ✅ [XAI-3] LIME (2 sampel) → E_lime_panel.png

  [4/4] Counterfactual...
   CF idx=4: Risiko Tinggi → Risiko Sedang ✅
   CF idx=5: Risiko Tinggi → Risiko Tinggi ⚠️
   CF idx=6: Risiko Tinggi → Risiko Tinggi ⚠️
   CF idx=0: Risiko Sedang → Risiko Sedang ⚠️
   CF idx=1: Risiko Sedang → Risiko Sedang ⚠️
   CF idx=2: Risiko Sedang → Risiko Sedang ⚠️
   Validasi CF-SHAP: 6/6 konsisten (100%)
   ✅ [XAI-4] Counterfactual tabel → E_counterfactual_table.png
   ✅ Ringkasan XAI 4-panel → E_xai_summary_4panel.png

  ✅ Semua XAI selesai (E_)

VERSI G — LOSO Temporal | Binary Classification

   Feature engineering TRAIN (LOSO)...
   ✅ 1248 siswa × 27 kolom
   Q75_train (binary threshold) = 3.7349

   Train: 1872 | Test: 899
   Distribusi test: [661 238]

🔍 Optuna Tuning Versi G (binary)...

Early stopping occurred a

  0%|          | 0/50 [00:00<?, ?it/s]

   Spearman ρ=0.7065 (p=0.0003) → VALID
   ✅ [XAI-2] SHAP summary beeswarm tersimpan
   ✅ [XAI-2] SHAP selesai (ρ=0.706)

  [3/4] LIME...
   ✅ [XAI-3] LIME (2 sampel) → G_lime_panel.png

  [4/4] Counterfactual...
   CF idx=42: Berisiko → Tidak Berisiko ✅
   CF idx=87: Berisiko → Tidak Berisiko ✅
   CF idx=107: Berisiko → Berisiko ⚠️
   Validasi CF-SHAP: 1/3 konsisten (33%)
   ✅ [XAI-4] Counterfactual tabel → G_counterfactual_table.png
   ✅ Ringkasan XAI 4-panel → G_xai_summary_4panel.png

  ✅ Semua XAI selesai (G_)

📊 RINGKASAN FINAL — SEMUA VERSI

  Versi                             TabNet F1   TabNet AUC      Best Model   SHAP ρ
  --------------------------------------------------------------------------------
  Versi B (StratKFold 5-fold, 3 kelas) 0.9653±0.0177 0.9971±0.0013    Improved DNN   0.8234
  Versi E (Temporal sem1-5/sem6)       0.8871       0.9665    Improved DNN   0.8688
  Versi G (LOSO Binary)                0.8218       0.9668         XGBoost   0.7065

📋 Prediksi semua 